In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl \
  scikit-learn pandas matplotlib seaborn sentencepiece safetensors

In [4]:
import os

PROJECT_DIR = "/content/drive/MyDrive/NimbusPay_Ticket_Triage"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/adapter", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/experiments", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/predictions", exist_ok=True)

print("Project directories created!")

for root, dirs, files in os.walk(PROJECT_DIR):
    print(root)

Project directories created!
/content/drive/MyDrive/NimbusPay_Ticket_Triage
/content/drive/MyDrive/NimbusPay_Ticket_Triage/data
/content/drive/MyDrive/NimbusPay_Ticket_Triage/adapter
/content/drive/MyDrive/NimbusPay_Ticket_Triage/experiments
/content/drive/MyDrive/NimbusPay_Ticket_Triage/predictions


In [6]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

data_files = [
    "train.jsonl",
    "dev.jsonl",
    "test_inputs.jsonl"
]

for filename, content in uploaded.items():

    if filename in data_files:
        destination = Path(PROJECT_DIR) / "data" / filename
    else:
        destination = Path(PROJECT_DIR) / filename

    destination.write_bytes(content)

    print("Uploaded:", filename)

print("\nAll uploaded files:")
print(os.listdir(PROJECT_DIR))
print("\nDataset files:")
print(os.listdir(f"{PROJECT_DIR}/data"))


Saving dev.jsonl to dev.jsonl
Saving test_inputs.jsonl to test_inputs.jsonl
Saving train.jsonl to train.jsonl
Uploaded: dev.jsonl
Uploaded: test_inputs.jsonl
Uploaded: train.jsonl

All uploaded files:
['data', 'adapter', 'experiments', 'predictions', 'score.py']

Dataset files:
['dev.jsonl', 'test_inputs.jsonl', 'train.jsonl']


In [7]:
import json

def load_jsonl(filename):

    path = f"{PROJECT_DIR}/data/{filename}"

    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl("train.jsonl")
dev_data = load_jsonl("dev.jsonl")
test_data = load_jsonl("test_inputs.jsonl")

print("========== DATASET SUMMARY ==========")

print("Training records:", len(train_data))
print("Development records:", len(dev_data))
print("Testing records:", len(test_data))

print("\n========== SAMPLE TRAINING RECORD ==========")

print(json.dumps(train_data[0], indent=2))

print("\n========== SETUP COMPLETE ==========")


========== DATASET SUMMARY ==========
Training records: 1445
Development records: 200
Testing records: 400

========== SAMPLE TRAINING RECORD ==========
{
  "id": "tr-00001",
  "messages": [
    {
      "role": "system",
      "content": "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."
    },
    {
      "role": "user",
      "content": "Received: 2026-07-25 (Sat)\nSource: app-chat\n\nHi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply."
    },
    {
      "role": "assistant",
      "content": "{\"category\": \"payment_failed\", \"priority\": \"P3\", \"amount_paise\": 470000, \"txn_date\": \"2026-07-25\", \"txn_id\": \"NP1599046332\", \"channel\": \"netbanking\", \"language\": \"en\", \"needs_human\": false}"
    }
  ]
}

========== SETUP COMPLETE ==========


In [2]:

!pip install -q --no-deps --force-reinstall \
    numpy==2.2.6 \
    pandas==2.2.3 \
    python-dateutil==2.9.0.post0


In [3]:

import torch
import numpy as np
import pandas as pd
import pyarrow
import dateutil

print("========== VERSION CHECK ==========")

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)
print("DateUtil:", dateutil.__version__)
print("PyTorch:", torch.__version__)

print("\n========== GPU CHECK ==========")

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n========== COMPLETE ==========")


========== VERSION CHECK ==========
NumPy: 2.4.6
Pandas: 2.2.3
PyArrow: 23.0.1
DateUtil: 2.9.0.post0
PyTorch: 2.14.1+cu130

========== GPU CHECK ==========
CUDA: True
GPU: Tesla T4

========== COMPLETE ==========


In [5]:


!pip install -q bitsandbytes


In [6]:

!pip install -q --no-deps --force-reinstall numpy==2.2.6


In [4]:

!pip install -q trl


In [5]:

import trl

print("TRL version:", trl.__version__)
print("TRL installed successfully!")


TRL version: 1.17.1
TRL installed successfully!


In [6]:

import torch
import numpy
import pandas
import pyarrow
import transformers
import datasets
import accelerate
import peft
import bitsandbytes
import trl

print("========== ENVIRONMENT ==========")

print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)

print("\n========== ML LIBRARIES ==========")

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)

print("\n========== GPU ==========")

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nALL CHECKS PASSED!")


========== ENVIRONMENT ==========
PyTorch: 2.14.1+cu130
NumPy: 2.2.6
Pandas: 2.2.3
PyArrow: 23.0.1

========== ML LIBRARIES ==========
Transformers: 5.22.1
Datasets: 4.8.5
Accelerate: 1.15.0
PEFT: 0.21.2
BitsAndBytes: 0.50.2
TRL: 1.17.1

========== GPU ==========
CUDA: True
GPU: Tesla T4

ALL CHECKS PASSED!


In [7]:

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import re
import pandas as pd
from collections import Counter, defaultdict

PROJECT_DIR = "/content/drive/MyDrive/NimbusPay_Ticket_Triage"
DATA_DIR = f"{PROJECT_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/train.jsonl"
DEV_PATH = f"{DATA_DIR}/dev.jsonl"
TEST_PATH = f"{DATA_DIR}/test_inputs.jsonl"
SCHEMA_PATH = f"{PROJECT_DIR}/SCHEMA.md"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl(TRAIN_PATH)
dev_data = load_jsonl(DEV_PATH)
test_data = load_jsonl(TEST_PATH)

print("Training:", len(train_data))
print("Development:", len(dev_data))
print("Testing:", len(test_data))

print("\nFirst training record:")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))


Mounted at /content/drive
Training: 1445
Development: 200
Testing: 400

First training record:
{
  "id": "tr-00001",
  "messages": [
    {
      "role": "system",
      "content": "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."
    },
    {
      "role": "user",
      "content": "Received: 2026-07-25 (Sat)\nSource: app-chat\n\nHi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply."
    },
    {
      "role": "assistant",
      "content": "{\"category\": \"payment_failed\", \"priority\": \"P3\", \"amount_paise\": 470000, \"txn_date\": \"2026-07-25\", \"txn_id\": \"NP1599046332\", \"channel\": \"netbanking\", \"language\": \"en\", \"needs_human\": false}"
    }
  ]
}


In [8]:

from collections import Counter, defaultdict
import json
import re

audit = {
    "total_records": len(train_data),
    "malformed_labels": [],
    "incorrect_priority_keys": [],
    "incorrect_dates": [],
    "incorrect_transaction_ids": [],
    "duplicate_messages": [],
    "invalid_categories": [],
    "invalid_channels": [],
    "invalid_languages": []
}

seen_messages = defaultdict(list)

VALID_CATEGORIES = {
    "payment_failed", "refund", "fraud",
    "account_access", "kyc", "offers", "other"
}

VALID_CHANNELS = {"upi", "card", "netbanking", "wallet"}
VALID_PRIORITIES = {"P1", "P2", "P3"}

for row in train_data:

    row_id = row["id"]
    messages = row["messages"]

    user_message = messages[1]["content"]
    assistant_output = messages[-1]["content"]

    # 1. Detect repeated customer messages
    seen_messages[user_message].append(row_id)

    # 2. Check whether assistant output is valid JSON
    try:
        label = json.loads(assistant_output)

        if not isinstance(label, dict):
            raise ValueError("Output is not a JSON object")

    except (json.JSONDecodeError, ValueError):
        audit["malformed_labels"].append(row_id)
        continue

    # 3. Detect incorrect priority key
    if "prio" in label and "priority" not in label:
        audit["incorrect_priority_keys"].append({
            "id": row_id,
            "value": label["prio"]
        })

    # 4. Check date formatting
    date = label.get("txn_date")

    if date is not None:
        if not isinstance(date, str) or not re.fullmatch(
            r"\d{4}-\d{2}-\d{2}", date
        ):
            audit["incorrect_dates"].append({
                "id": row_id,
                "value": date
            })

    # 5. Check transaction ID formatting
    txn_id = label.get("txn_id")

    if txn_id is not None:
        if not isinstance(txn_id, str) or not re.fullmatch(
            r"NP\d{10}", txn_id
        ):
            audit["incorrect_transaction_ids"].append({
                "id": row_id,
                "value": txn_id
            })

    # 6. Check categories
    if label.get("category") not in VALID_CATEGORIES:
        audit["invalid_categories"].append(row_id)

    # 7. Check channels
    if label.get("channel") not in VALID_CHANNELS | {None}:
        audit["invalid_channels"].append(row_id)

    # 8. Check languages
    if label.get("language") not in {"en", "hi-en"}:
        audit["invalid_languages"].append(row_id)


# Collect duplicate groups
for message, ids in seen_messages.items():
    if len(ids) > 1:
        audit["duplicate_messages"].append(ids)


# Display results
print("=" * 55)
print("       NIMBUSPAY TRAINING DATA AUDIT")
print("=" * 55)

print("Total records:", audit["total_records"])

print("\nDATA QUALITY ISSUES")

for key, value in audit.items():
    if key != "total_records":
        print(f"{key}: {len(value)}")

print("\nPRIORITY VALUE DISTRIBUTION")

priority_values = Counter()

for row in train_data:
    try:
        label = json.loads(row["messages"][-1]["content"])
        if "prio" in label:
            priority_values[str(label["prio"])] += 1
        elif "priority" in label:
            priority_values[str(label["priority"])] += 1
    except:
        continue

print(priority_values)

print("\nDUPLICATE MESSAGE GROUPS:",
      len(audit["duplicate_messages"]))

print("\nAUDIT COMPLETED")


       NIMBUSPAY TRAINING DATA AUDIT
Total records: 1445

DATA QUALITY ISSUES
malformed_labels: 25
incorrect_priority_keys: 70
incorrect_dates: 30
incorrect_transaction_ids: 20
duplicate_messages: 60
invalid_categories: 0
invalid_channels: 0
invalid_languages: 0

PRIORITY VALUE DISTRIBUTION
Counter({'P3': 744, 'P2': 320, 'P1': 286, 'low': 44, 'medium': 15, 'high': 11})

DUPLICATE MESSAGE GROUPS: 60

AUDIT COMPLETED


In [9]:
print("=" * 60)
print("DETAILED DATA AUDIT")
print("=" * 60)

print("\n1. MALFORMED LABELS")
for row_id in audit["malformed_labels"][:10]:
    row = next(r for r in train_data if r["id"] == row_id)
    print("\nID:", row_id)
    print("Raw label:", row["messages"][-1]["content"])

print("\n2. INCORRECT PRIORITY KEYS")
for item in audit["incorrect_priority_keys"][:10]:
    print("\nID:", item["id"])
    print("Value:", item["value"])

print("\n3. INCORRECT DATES")
for item in audit["incorrect_dates"][:10]:
    print(item)

print("\n4. INCORRECT TRANSACTION IDS")
for item in audit["incorrect_transaction_ids"][:10]:
    print(item)

print("\n5. DUPLICATE MESSAGE GROUPS")
for group in audit["duplicate_messages"][:10]:
    print(group)

print("\n" + "=" * 60)
print("DETAILED AUDIT COMPLETED")

DETAILED DATA AUDIT

1. MALFORMED LABELS

ID: tr-00060
Raw label: {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "t

ID: tr-00095
Raw label: {"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "c

ID: tr-00125
Raw label: {"category": "payment_failed", "priority": "P3", "amount_paise": 392500, "txn_date": "2026-08-12", "t

ID: tr-00172
Raw label: {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_i

ID: tr-00267
Raw label: {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channe

ID: tr-00357
Raw label: {"category": "account_access", "priority": "P2", "amount_paise": null, "txn_date": null, "txn_id": null, "channel

ID: tr-00478
Raw label: {"category": "refund", "priority": "P1", "amount_paise": 16041600, "txn_date": null, "txn_id": "NP1893758181", "channe

ID: tr-00497
Raw label: {"category": "payment_failed", "priority": "P3", "amount_

In [10]:
import json

def get_record(row_id):
    return next(row for row in train_data if row["id"] == row_id)

def get_message(row, role):
    return next(
        message["content"]
        for message in row["messages"]
        if message["role"] == role
    )

print("=" * 70)
print("1. FULL MALFORMED RECORDS")
print("=" * 70)

for row_id in audit["malformed_labels"]:
    row = get_record(row_id)

    print(f"\nID: {row_id}")
    print("TICKET:")
    print(get_message(row, "user"))
    print("\nRAW LABEL:")
    print(repr(get_message(row, "assistant")))
    print("-" * 70)


print("\n2. PRIORITY LABEL INSPECTION")
print("=" * 70)

for item in audit["incorrect_priority_keys"][:10]:
    row = get_record(item["id"])

    print(f"\nID: {item['id']}")
    print("TICKET:")
    print(get_message(row, "user"))
    print("LABEL:")
    print(get_message(row, "assistant"))
    print("-" * 70)


print("\n3. DATE INSPECTION")
print("=" * 70)

for item in audit["incorrect_dates"][:10]:
    row = get_record(item["id"])

    print(f"\nID: {item['id']}")
    print("TICKET:")
    print(get_message(row, "user"))
    print("LABEL:")
    print(get_message(row, "assistant"))
    print("-" * 70)


print("\n4. TRANSACTION ID INSPECTION")
print("=" * 70)

for item in audit["incorrect_transaction_ids"][:10]:
    row = get_record(item["id"])

    print(f"\nID: {item['id']}")
    print("TICKET:")
    print(get_message(row, "user"))
    print("LABEL:")
    print(get_message(row, "assistant"))
    print("-" * 70)


print("\n5. DUPLICATE LABEL COMPARISON")
print("=" * 70)

for group in audit["duplicate_messages"][:10]:
    row1 = get_record(group[0])
    row2 = get_record(group[1])

    label1 = get_message(row1, "assistant")
    label2 = get_message(row2, "assistant")

    print(f"\nRecords: {group}")
    print("Labels identical:", label1 == label2)
    print("First label:", label1)
    print("Second label:", label2)
    print("-" * 70)

print("\nINSPECTION COMPLETED")

1. FULL MALFORMED RECORDS

ID: tr-00060
TICKET:
Received: 2026-09-17 (Thu)
Source: email

Subject: Complaint

Hi team,
Everytime I try to prove my identity it won't let me. Bank ref no 974846156473.

Regards,
Ananya Bose

RAW LABEL:
'{"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "t'
----------------------------------------------------------------------

ID: tr-00095
TICKET:
Received: 2026-09-15 (Tue)
Source: app-chat

Dear NimbusPay support, May I have another card? My phone is a Realme 11. Thanks.

RAW LABEL:
'{"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "c'
----------------------------------------------------------------------

ID: tr-00125
TICKET:
Received: 2026-08-18 (Tue)
Source: app-chat

Hey, My card payment didn't work. Dated 12-08-26. It is Rs. 3925 in total. The txn id is NP6948115654. My earlier ticket number is #99485. Kindly resolve this asap.

RAW LABEL:
'{"category": "payment_failed", "pri

In [11]:
print("=" * 60)
print("DUPLICATE CONSISTENCY CHECK")
print("=" * 60)

identical_label_groups = []
conflicting_label_groups = []

for group in audit["duplicate_messages"]:
    labels = []

    for row_id in group:
        row = get_record(row_id)
        raw_label = get_message(row, "assistant")

        try:
            labels.append(json.loads(raw_label))
        except json.JSONDecodeError:
            labels.append(raw_label)

    if all(label == labels[0] for label in labels):
        identical_label_groups.append(group)
    else:
        conflicting_label_groups.append(group)

print("Total duplicate groups:", len(audit["duplicate_messages"]))
print("Groups with identical labels:", len(identical_label_groups))
print("Groups with conflicting labels:", len(conflicting_label_groups))

print("\nCONFLICTING GROUPS")
for group in conflicting_label_groups:
    print("\nIDs:", group)

    for row_id in group:
        row = get_record(row_id)
        print("ID:", row_id)
        print("Ticket:", get_message(row, "user"))
        print("Label:", get_message(row, "assistant"))

print("\n" + "=" * 60)
print("DUPLICATE CHECK COMPLETED")

DUPLICATE CONSISTENCY CHECK
Total duplicate groups: 60
Groups with identical labels: 60
Groups with conflicting labels: 0

CONFLICTING GROUPS

DUPLICATE CHECK COMPLETED


In [12]:
import csv
import json
import os

# Preserve original training data
clean_data = train_data.copy()

# Identify duplicate records to remove
duplicate_ids_to_drop = set()

for group in identical_label_groups:
    # Keep the first record, remove the remaining copies
    duplicate_ids_to_drop.update(group[1:])

# Create cleaned working dataset
clean_data = [
    row for row in clean_data
    if row["id"] not in duplicate_ids_to_drop
]

# Record every removal
cleaning_log = []

for row_id in sorted(duplicate_ids_to_drop):
    cleaning_log.append({
        "id": row_id,
        "action": "drop",
        "reason": "Exact duplicate ticket with identical label; retained first occurrence."
    })

# Save intermediate dataset
DEDUP_PATH = f"{DATA_DIR}/train_deduplicated.jsonl"

with open(DEDUP_PATH, "w", encoding="utf-8") as f:
    for row in clean_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Save cleaning log
LOG_PATH = f"{PROJECT_DIR}/cleaning_log.csv"

with open(LOG_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["id", "action", "reason"]
    )
    writer.writeheader()
    writer.writerows(cleaning_log)

print("DUPLICATE CLEANING COMPLETED")
print("-" * 45)
print("Original records:", len(train_data))
print("Removed duplicate records:", len(duplicate_ids_to_drop))
print("Remaining records:", len(clean_data))
print("Cleaning log entries:", len(cleaning_log))
print("Dataset saved:", DEDUP_PATH)
print("Cleaning log saved:", LOG_PATH)

DUPLICATE CLEANING COMPLETED
---------------------------------------------
Original records: 1445
Removed duplicate records: 60
Remaining records: 1385
Cleaning log entries: 60
Dataset saved: /content/drive/MyDrive/NimbusPay_Ticket_Triage/data/train_deduplicated.jsonl
Cleaning log saved: /content/drive/MyDrive/NimbusPay_Ticket_Triage/cleaning_log.csv


In [13]:
import json

legacy_priorities = {"low", "medium", "high"}

priority_records = []

for row in clean_data:
    try:
        label = json.loads(row["messages"][-1]["content"])

        priority = label.get("priority", label.get("prio"))

        if priority in legacy_priorities:
            priority_records.append({
                "id": row["id"],
                "priority": priority,
                "ticket": row["messages"][1]["content"],
                "label": label
            })

    except json.JSONDecodeError:
        continue

print("Remaining legacy priority records:", len(priority_records))

for item in priority_records:
    print("\nID:", item["id"])
    print("Old priority:", item["priority"])
    print("Ticket:", item["ticket"])
    print("Label:", item["label"])
    print("-" * 60)

Remaining legacy priority records: 70

ID: tr-00020
Old priority: low
Ticket: Received: 2026-09-05 (Sat)
Source: ivr-transcript

[IVR transcript] Hey, How can I check if I have received a refund if it is not yet on my statement? 978 rupees is the amount. I have contacted support about this before. Mode: wallet. Using the Android app. This happened on 02-09-26. Thanks.
Label: {'category': 'refund', 'prio': 'low', 'amount_paise': 97800, 'txn_date': '02/09/2026', 'txn_id': None, 'channel': 'wallet', 'language': 'en', 'needs_human': True}
------------------------------------------------------------

ID: tr-00049
Old priority: low
Ticket: Received: 2026-09-01 (Tue)
Source: ivr-transcript

[IVR transcript]  Regsitered email id kaise change karun? Dhanyavaad.
Label: {'category': 'other', 'prio': 'low', 'amount_paise': None, 'txn_date': None, 'txn_id': None, 'channel': None, 'language': 'hi-en', 'needs_human': False}
------------------------------------------------------------

ID: tr-00058
Ol

In [14]:
import re
import json
import os
import csv

PRIORITY_OUTPUT = f"{DATA_DIR}/train_priority_fixed.jsonl"

LEGAL_PATTERN = re.compile(
    r"\b(RBI|ombudsman|consumer court|lawyer|legal notice|legal action)\b",
    re.IGNORECASE
)

CONTACT_PATTERN = re.compile(
    r"contacted support before|already contacted|"
    r"second complaint|third time|fourth time|"
    r"second time|third time|fourth time|"
    r"already written \d+ times|written \d+ times|"
    r"again and again|baar baar|pehle bhi complaint|"
    r"pichla ticket|earlier ticket|previous ticket|"
    r"already complained|complained before",
    re.IGNORECASE
)

TRANSACTION_CATEGORIES = {"payment_failed", "refund"}

priority_changes = 0
human_flag_changes = 0
malformed_skipped = 0

for row in clean_data:
    raw_label = row["messages"][-1]["content"]

    try:
        label = json.loads(raw_label)
    except json.JSONDecodeError:
        malformed_skipped += 1
        continue

    old_label = label.copy()

    # Migrate the incorrect key
    if "prio" in label:
        if "priority" not in label:
            label["priority"] = label["prio"]
        del label["prio"]

    ticket = row["messages"][1]["content"]

    category = label.get("category")
    amount = label.get("amount_paise")

    try:
        amount = float(amount) if amount is not None else 0
    except (ValueError, TypeError):
        amount = 0

    # Recalculate priority using the assignment schema
    legal_threat = bool(LEGAL_PATTERN.search(ticket))

    if (
        category == "fraud"
        or (category in TRANSACTION_CATEGORIES and amount >= 5_000_000)
        or legal_threat
    ):
        new_priority = "P1"

    elif (
        category == "account_access"
        or (category in TRANSACTION_CATEGORIES and amount >= 500_000)
    ):
        new_priority = "P2"

    else:
        new_priority = "P3"

    label["priority"] = new_priority

    # needs_human is independent of priority
    previous_contact = bool(CONTACT_PATTERN.search(ticket))
    label["needs_human"] = (
        new_priority == "P1" or previous_contact
    )

    # Record changes
    if label != old_label:
        reasons = []

        if "prio" in old_label:
            reasons.append("Migrated incorrect prio key to priority")

        if old_label.get("priority") != new_priority:
            reasons.append(
                f"Recalculated priority as {new_priority} using schema rules"
            )
            priority_changes += 1

        if old_label.get("needs_human") != label["needs_human"]:
            reasons.append("Corrected needs_human using escalation rules")
            human_flag_changes += 1

        cleaning_log.append({
            "id": row["id"],
            "action": "fix",
            "reason": "; ".join(reasons)
        })

        row["messages"][-1]["content"] = json.dumps(
            label, ensure_ascii=False
        )

# Save the updated dataset
with open(PRIORITY_OUTPUT, "w", encoding="utf-8") as f:
    for row in clean_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Update cleaning log
with open(LOG_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["id", "action", "reason"]
    )
    writer.writeheader()
    writer.writerows(cleaning_log)

print("=" * 55)
print("PRIORITY CLEANING COMPLETED")
print("=" * 55)
print("Priority changes:", priority_changes)
print("Human escalation flag changes:", human_flag_changes)
print("Malformed records skipped:", malformed_skipped)
print("Total records:", len(clean_data))
print("Cleaning log entries:", len(cleaning_log))
print("Saved dataset:", PRIORITY_OUTPUT)
print("Saved log:", LOG_PATH)

print("\nFINAL PRIORITY DISTRIBUTION")

from collections import Counter

priority_counts = Counter()

for row in clean_data:
    try:
        label = json.loads(row["messages"][-1]["content"])
        priority_counts[label["priority"]] += 1
    except json.JSONDecodeError:
        continue

print(priority_counts)

PRIORITY CLEANING COMPLETED
Priority changes: 135
Human escalation flag changes: 93
Malformed records skipped: 25
Total records: 1385
Cleaning log entries: 280
Saved dataset: /content/drive/MyDrive/NimbusPay_Ticket_Triage/data/train_priority_fixed.jsonl
Saved log: /content/drive/MyDrive/NimbusPay_Ticket_Triage/cleaning_log.csv

FINAL PRIORITY DISTRIBUTION
Counter({'P3': 767, 'P2': 331, 'P1': 262})


In [15]:
import re
import json
import csv
from datetime import datetime

date_fixes = 0
txn_id_fixes = 0
invalid_dates = []
invalid_txn_ids = []

for row in clean_data:
    try:
        label = json.loads(row["messages"][-1]["content"])
    except json.JSONDecodeError:
        continue

    old_date = label.get("txn_date")
    old_txn_id = label.get("txn_id")

    # Normalize dates from DD/MM/YYYY to YYYY-MM-DD
    if isinstance(old_date, str):
        try:
            parsed_date = datetime.strptime(old_date, "%d/%m/%Y")
            new_date = parsed_date.strftime("%Y-%m-%d")

            if new_date != old_date:
                label["txn_date"] = new_date
                date_fixes += 1

                cleaning_log.append({
                    "id": row["id"],
                    "action": "fix",
                    "reason": f"Normalized transaction date from {old_date} to {new_date}"
                })

        except ValueError:
            try:
                datetime.strptime(old_date, "%Y-%m-%d")
            except ValueError:
                invalid_dates.append({
                    "id": row["id"],
                    "value": old_date
                })

    # Normalize transaction IDs
    if isinstance(old_txn_id, str):
        normalized_id = re.sub(
            r"[^A-Za-z0-9]",
            "",
            old_txn_id
        ).upper()

        if re.fullmatch(r"NP\d{10}", normalized_id):
            if normalized_id != old_txn_id:
                label["txn_id"] = normalized_id
                txn_id_fixes += 1

                cleaning_log.append({
                    "id": row["id"],
                    "action": "fix",
                    "reason": f"Normalized transaction ID from {old_txn_id} to {normalized_id}"
                })
        else:
            invalid_txn_ids.append({
                "id": row["id"],
                "value": old_txn_id
            })

    row["messages"][-1]["content"] = json.dumps(
        label,
        ensure_ascii=False
    )

# Save updated dataset
DATE_TXN_PATH = f"{DATA_DIR}/train_dates_txn_fixed.jsonl"

with open(DATE_TXN_PATH, "w", encoding="utf-8") as f:
    for row in clean_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Update cleaning log
with open(LOG_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["id", "action", "reason"]
    )
    writer.writeheader()
    writer.writerows(cleaning_log)

print("=" * 55)
print("DATE AND TRANSACTION ID CLEANING")
print("=" * 55)
print("Date fixes:", date_fixes)
print("Transaction ID fixes:", txn_id_fixes)
print("Invalid dates remaining:", len(invalid_dates))
print("Invalid transaction IDs remaining:", len(invalid_txn_ids))
print("Cleaning log entries:", len(cleaning_log))
print("Saved dataset:", DATE_TXN_PATH)
print("Saved log:", LOG_PATH)

print("\nINVALID DATES:")
print(invalid_dates[:10])

print("\nINVALID TRANSACTION IDS:")
print(invalid_txn_ids[:10])

DATE AND TRANSACTION ID CLEANING
Date fixes: 30
Transaction ID fixes: 20
Invalid dates remaining: 0
Invalid transaction IDs remaining: 0
Cleaning log entries: 330
Saved dataset: /content/drive/MyDrive/NimbusPay_Ticket_Triage/data/train_dates_txn_fixed.jsonl
Saved log: /content/drive/MyDrive/NimbusPay_Ticket_Triage/cleaning_log.csv

INVALID DATES:
[]

INVALID TRANSACTION IDS:
[]


In [16]:
import json
import csv

valid_data = []
malformed_ids = []

for row in clean_data:
    raw_label = row["messages"][-1]["content"]

    try:
        label = json.loads(raw_label)

        if not isinstance(label, dict):
            raise ValueError("Label is not a JSON object")

        valid_data.append(row)

    except (json.JSONDecodeError, ValueError):
        malformed_ids.append(row["id"])

# Document every removal
for row_id in malformed_ids:
    cleaning_log.append({
        "id": row_id,
        "action": "drop",
        "reason": "Truncated malformed JSON label; complete output could not be validated reliably."
    })

# Replace working dataset
clean_data = valid_data

# Save cleaned dataset
CLEAN_PATH = f"{DATA_DIR}/train_cleaned.jsonl"

with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    for row in clean_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Update cleaning log
with open(LOG_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["id", "action", "reason"]
    )
    writer.writeheader()
    writer.writerows(cleaning_log)

print("=" * 55)
print("MALFORMED LABEL CLEANING COMPLETED")
print("=" * 55)
print("Malformed records removed:", len(malformed_ids))
print("Remaining training records:", len(clean_data))
print("Cleaning log entries:", len(cleaning_log))
print("Dataset:", CLEAN_PATH)
print("Cleaning log:", LOG_PATH)

MALFORMED LABEL CLEANING COMPLETED
Malformed records removed: 25
Remaining training records: 1360
Cleaning log entries: 355
Dataset: /content/drive/MyDrive/NimbusPay_Ticket_Triage/data/train_cleaned.jsonl
Cleaning log: /content/drive/MyDrive/NimbusPay_Ticket_Triage/cleaning_log.csv


In [17]:
import json
import re
from datetime import datetime
from collections import Counter, defaultdict

REQUIRED_KEYS = {
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
}

VALID_CATEGORIES = {
    "payment_failed", "refund", "fraud",
    "account_access", "kyc", "offers", "other"
}

VALID_PRIORITIES = {"P1", "P2", "P3"}
VALID_CHANNELS = {"upi", "card", "netbanking", "wallet"}
VALID_LANGUAGES = {"en", "hi-en"}

NULL_FIELDS_CATEGORIES = {"account_access", "kyc", "other"}

errors = defaultdict(list)
priority_distribution = Counter()
category_distribution = Counter()

for row in clean_data:
    row_id = row.get("id")

    try:
        messages = row["messages"]
        user_message = next(
            m["content"] for m in messages if m["role"] == "user"
        )
        raw_label = next(
            m["content"] for m in messages if m["role"] == "assistant"
        )
        label = json.loads(raw_label)

    except Exception as e:
        errors["invalid_record_structure"].append(
            {"id": row_id, "error": str(e)}
        )
        continue

    if not isinstance(label, dict):
        errors["label_not_object"].append(row_id)
        continue

    # 1. Exact key validation
    if set(label.keys()) != REQUIRED_KEYS:
        errors["incorrect_keys"].append({
            "id": row_id,
            "missing": sorted(REQUIRED_KEYS - set(label.keys())),
            "extra": sorted(set(label.keys()) - REQUIRED_KEYS)
        })

    # 2. Category
    category = label.get("category")
    if category not in VALID_CATEGORIES:
        errors["invalid_category"].append(row_id)
    else:
        category_distribution[category] += 1

    # 3. Priority
    priority = label.get("priority")
    if priority not in VALID_PRIORITIES:
        errors["invalid_priority"].append(row_id)
    else:
        priority_distribution[priority] += 1

    # 4. Amount
    amount = label.get("amount_paise")
    if amount is not None:
        if type(amount) is not int or amount < 0:
            errors["invalid_amount"].append({
                "id": row_id, "value": amount
            })

    # 5. Date
    date = label.get("txn_date")
    if date is not None:
        try:
            if not isinstance(date, str):
                raise ValueError("Date is not a string")
            datetime.strptime(date, "%Y-%m-%d")
        except ValueError:
            errors["invalid_date"].append({
                "id": row_id, "value": date
            })

    # 6. Transaction ID
    txn_id = label.get("txn_id")
    if txn_id is not None:
        if not isinstance(txn_id, str) or not re.fullmatch(
            r"NP\d{10}", txn_id
        ):
            errors["invalid_txn_id"].append({
                "id": row_id, "value": txn_id
            })

    # 7. Channel
    channel = label.get("channel")
    if channel is not None and channel not in VALID_CHANNELS:
        errors["invalid_channel"].append({
            "id": row_id, "value": channel
        })

    # 8. Language
    if label.get("language") not in VALID_LANGUAGES:
        errors["invalid_language"].append({
            "id": row_id, "value": label.get("language")
        })

    # 9. needs_human must be a boolean
    if type(label.get("needs_human")) is not bool:
        errors["invalid_needs_human"].append({
            "id": row_id, "value": label.get("needs_human")
        })

    # 10. P1 must always require human review
    if priority == "P1" and label.get("needs_human") is not True:
        errors["p1_without_human"].append(row_id)

    # 11. Transaction fields must be null for specified categories
    if category in NULL_FIELDS_CATEGORIES:
        for field in ["amount_paise", "txn_date", "txn_id", "channel"]:
            if label.get(field) is not None:
                errors["non_null_transaction_fields"].append({
                    "id": row_id,
                    "category": category,
                    "field": field,
                    "value": label[field]
                })

print("=" * 65)
print("FINAL TRAINING DATA SCHEMA VALIDATION")
print("=" * 65)

print("Total records checked:", len(clean_data))

print("\nISSUE COUNTS")
total_issues = 0

for issue, records in errors.items():
    print(f"{issue}: {len(records)}")
    total_issues += len(records)

if not errors:
    print("No structural schema errors detected.")

print("\nPRIORITY DISTRIBUTION")
print(dict(priority_distribution))

print("\nCATEGORY DISTRIBUTION")
print(dict(category_distribution))

print("\nTOTAL VALIDATION ISSUES:", total_issues)

print("\nFIRST 20 ISSUES")
for issue, records in errors.items():
    for record in records[:20]:
        print(issue, record)

print("\n" + "=" * 65)
print("VALIDATION COMPLETED")
print("=" * 65)

FINAL TRAINING DATA SCHEMA VALIDATION
Total records checked: 1360

ISSUE COUNTS
No structural schema errors detected.

PRIORITY DISTRIBUTION
{'P3': 767, 'P2': 331, 'P1': 262}

CATEGORY DISTRIBUTION
{'payment_failed': 395, 'account_access': 174, 'refund': 301, 'fraud': 84, 'kyc': 163, 'offers': 145, 'other': 98}

TOTAL VALIDATION ISSUES: 0

FIRST 20 ISSUES

VALIDATION COMPLETED


In [18]:
import re
import json
import csv

PREVIOUS_CONTACT_PATTERNS = [
    r"\bcontacted support\b.{0,60}\b(before|previously|again)\b",
    r"\b(?:i have|i've)\s+(?:already\s+)?contacted support\b",
    r"\b(?:second|third|fourth)\s+time\b",
    r"\b(?:second|third|fourth)\s+complaint\b",
    r"\b(?:already|previously)\s+(?:contacted|written|emailed|complained)\b",
    r"\b(?:written|emailed|contacted)\s+\d+\s+times\b",
    r"\bagain and again\b",
    r"\baar baar\b",
    r"\bpehle bhi complaint\b",
    r"\bcomplained before\b",
    r"\b(?:earlier|previous) ticket\b"
]

previous_contact_regex = re.compile(
    "|".join(PREVIOUS_CONTACT_PATTERNS),
    re.IGNORECASE
)

corrected_flags = []
detected_contacts = 0

for row in clean_data:
    label = json.loads(row["messages"][-1]["content"])
    ticket = row["messages"][1]["content"]

    previous_contact = bool(
        previous_contact_regex.search(ticket)
    )

    if previous_contact:
        detected_contacts += 1

    should_be_human = (
        label["priority"] == "P1" or previous_contact
    )

    if should_be_human and label["needs_human"] is not True:
        label["needs_human"] = True

        corrected_flags.append(row["id"])

        cleaning_log.append({
            "id": row["id"],
            "action": "fix",
            "reason": "Corrected needs_human to true because ticket indicates previous support contact or P1 priority."
        })

        row["messages"][-1]["content"] = json.dumps(
            label,
            ensure_ascii=False
        )

# Save corrected dataset
with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    for row in clean_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Save updated cleaning log
with open(LOG_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["id", "action", "reason"]
    )
    writer.writeheader()
    writer.writerows(cleaning_log)

print("=" * 60)
print("FINAL ESCALATION AUDIT")
print("=" * 60)
print("Records checked:", len(clean_data))
print("Tickets indicating previous contact:", detected_contacts)
print("Incorrect flags corrected:", len(corrected_flags))
print("P1 records:", sum(
    json.loads(r["messages"][-1]["content"])["priority"] == "P1"
    for r in clean_data
))
print("Records requiring human review:", sum(
    json.loads(r["messages"][-1]["content"])["needs_human"] is True
    for r in clean_data
))
print("Total cleaning log entries:", len(cleaning_log))
print("Final dataset:", CLEAN_PATH)
print("Cleaning log:", LOG_PATH)

FINAL ESCALATION AUDIT
Records checked: 1360
Tickets indicating previous contact: 182
Incorrect flags corrected: 19
P1 records: 262
Records requiring human review: 441
Total cleaning log entries: 374
Final dataset: /content/drive/MyDrive/NimbusPay_Ticket_Triage/data/train_cleaned.jsonl
Cleaning log: /content/drive/MyDrive/NimbusPay_Ticket_Triage/cleaning_log.csv


In [19]:
import json
import os

final_records = []

with open(CLEAN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            final_records.append(json.loads(line))

invalid = []

for row in final_records:
    try:
        label = json.loads(row["messages"][-1]["content"])

        expected_human = (
            label["priority"] == "P1"
            or bool(previous_contact_regex.search(
                row["messages"][1]["content"]
            ))
        )

        if label["needs_human"] != expected_human:
            invalid.append(row["id"])

    except Exception:
        invalid.append(row["id"])

print("=" * 55)
print("FINAL DATASET VERIFICATION")
print("=" * 55)
print("Records:", len(final_records))
print("Incorrect escalation flags:", len(invalid))
print("File exists:", os.path.exists(CLEAN_PATH))
print("Cleaning log exists:", os.path.exists(LOG_PATH))
print("Verification:", "PASSED" if not invalid else "FAILED")

if invalid:
    print("Problematic IDs:", invalid[:20])

FINAL DATASET VERIFICATION
Records: 1360
Incorrect escalation flags: 27
File exists: True
Cleaning log exists: True
Verification: FAILED
Problematic IDs: ['tr-00005', 'tr-00222', 'tr-00322', 'tr-00361', 'tr-00369', 'tr-00413', 'tr-00475', 'tr-00504', 'tr-00518', 'tr-00586', 'tr-00673', 'tr-00736', 'tr-00767', 'tr-00770', 'tr-00855', 'tr-00882', 'tr-00885', 'tr-00972', 'tr-01115', 'tr-01121']


In [20]:
import json

print("=" * 65)
print("ESCALATION FLAG MISMATCH DIAGNOSTICS")
print("=" * 65)

for row_id in invalid:
    row = next(r for r in final_records if r["id"] == row_id)

    label = json.loads(row["messages"][-1]["content"])
    ticket = row["messages"][1]["content"]

    print("\nID:", row_id)
    print("Priority:", label["priority"])
    print("Current needs_human:", label["needs_human"])
    print("Previous-contact pattern matched:",
          bool(previous_contact_regex.search(ticket)))
    print("Ticket:")
    print(ticket)
    print("-" * 65)

ESCALATION FLAG MISMATCH DIAGNOSTICS

ID: tr-00005
Priority: P2
Current needs_human: True
Previous-contact pattern matched: False
Ticket:
Received: 2026-06-22 (Mon)
Source: twitter

@NimbusPayCare OTP nahi aa raha isliye sign in nahi kar pa raha. Pichla ticket number #88470 tha.
-----------------------------------------------------------------

ID: tr-00222
Priority: P3
Current needs_human: True
Previous-contact pattern matched: False
Ticket:
Received: 2026-09-19 (Sat)
Source: email

Subject: Need support


Saamaan wapas kar diya, ab tak paise wapas nahi mile. 13-09-26 ki transaction hai. Total Rs 594.50 ka tha. Pichla ticket number #84647 tha.

Regards,
Pooja Nair
-----------------------------------------------------------------

ID: tr-00322
Priority: P3
Current needs_human: True
Previous-contact pattern matched: False
Ticket:
Received: 2026-09-07 (Mon)
Source: app-chat

Bhai, Booking cancel ki thi, refund ka koi ata pata nahi. Amount Rs. 1132 tha. Baar baar complaint kar raha hoon. 

In [21]:
import re
import json

CONTACT_PATTERNS = [
    r"\bpichla\s+ticket\b",
    r"\bprevious\s+ticket\b",
    r"\bearlier\s+ticket\b",
    r"\b(?:baar|bar)\s+(?:baar|bar)\b",
    r"\b(?:teesri|tisri|third)\s+baar\b",
    r"\b(?:second|third|fourth)\s+time\b",
    r"\b(?:second|third|fourth)\s+complaint\b",
    r"\bcontacted support before\b",
    r"\b(?:i have|i've)\s+(?:already\s+)?contacted support\b",
    r"\b(?:already|previously)\s+(?:contacted|written|emailed|complained)\b",
    r"\b(?:written|emailed|contacted)\s+\d+\s+times\b",
    r"\bagain and again\b",
    r"\bpehle bhi complaint\b",
    r"\bcomplained before\b"
]

contact_regex = re.compile(
    "|".join(CONTACT_PATTERNS),
    re.IGNORECASE
)

def remove_quoted_lines(ticket):
    return "\n".join(
        line for line in ticket.splitlines()
        if not line.strip().startswith(">")
    )

verification_errors = []

for row in final_records:
    label = json.loads(row["messages"][-1]["content"])
    ticket = remove_quoted_lines(row["messages"][1]["content"])

    previous_contact = bool(contact_regex.search(ticket))

    expected = (
        label["priority"] == "P1"
        or previous_contact
    )

    if label["needs_human"] != expected:
        verification_errors.append({
            "id": row["id"],
            "priority": label["priority"],
            "current": label["needs_human"],
            "expected": expected
        })

print("=" * 60)
print("CORRECTED ESCALATION VERIFICATION")
print("=" * 60)
print("Records checked:", len(final_records))
print("Remaining mismatches:", len(verification_errors))
print("Verification:", "PASSED" if not verification_errors else "FAILED")

if verification_errors:
    print("\nMISMATCHES:")
    for item in verification_errors[:30]:
        print(item)

CORRECTED ESCALATION VERIFICATION
Records checked: 1360
Remaining mismatches: 13
Verification: FAILED

MISMATCHES:
{'id': 'tr-00031', 'priority': 'P2', 'current': False, 'expected': True}
{'id': 'tr-00050', 'priority': 'P3', 'current': False, 'expected': True}
{'id': 'tr-00072', 'priority': 'P3', 'current': False, 'expected': True}
{'id': 'tr-00107', 'priority': 'P3', 'current': False, 'expected': True}
{'id': 'tr-00246', 'priority': 'P3', 'current': False, 'expected': True}
{'id': 'tr-00341', 'priority': 'P3', 'current': False, 'expected': True}
{'id': 'tr-00362', 'priority': 'P3', 'current': False, 'expected': True}
{'id': 'tr-00541', 'priority': 'P3', 'current': False, 'expected': True}
{'id': 'tr-00901', 'priority': 'P2', 'current': False, 'expected': True}
{'id': 'tr-00919', 'priority': 'P2', 'current': False, 'expected': True}
{'id': 'tr-01242', 'priority': 'P3', 'current': False, 'expected': True}
{'id': 'tr-01271', 'priority': 'P2', 'current': False, 'expected': True}
{'id': 't

In [22]:
print("=" * 65)
print("FINAL 13 ESCALATION MISMATCHES")
print("=" * 65)

for item in verification_errors:
    row = next(
        r for r in final_records
        if r["id"] == item["id"]
    )

    print("\nID:", item["id"])
    print("Priority:", item["priority"])
    print("Current needs_human:", item["current"])
    print("Expected needs_human:", item["expected"])
    print("Ticket:")
    print(remove_quoted_lines(row["messages"][1]["content"]))
    print("-" * 65)

FINAL 13 ESCALATION MISMATCHES

ID: tr-00031
Priority: P2
Current needs_human: False
Expected needs_human: True
Ticket:
Received: 2026-08-11 (Tue)
Source: app-chat

Return accept ho gaya par refund nahi aaya abhi tak. App version 6.8.14 hai. Transaction date 05-08-26 hai. Teesri baar bol raha hoon. Amount ₹44,348 tha. Please help karo.
-----------------------------------------------------------------

ID: tr-00050
Priority: P3
Current needs_human: False
Expected needs_human: True
Ticket:
Received: 2026-07-03 (Fri)
Source: app-chat

Sir, Srcatch card ka reawrd add nahi hua. Bank ref 325733031587 hai. Teesri baar bol raha hoon. Dhanyavaad.
-----------------------------------------------------------------

ID: tr-00072
Priority: P3
Current needs_human: False
Expected needs_human: True
Ticket:
Received: 2026-06-01 (Mon)
Source: ivr-transcript

[IVR transcript] Hello ji, PAN verfiication fail ho raha hai KYC mein, detalis sahi hain. Teesri baar bol raha hoon. Bank ref 804327892092 hai.
----

In [24]:
CLEANING_LOG_PATH = f"{PROJECT_DIR}/cleaning_log.csv"

print("Cleaning log path:", CLEANING_LOG_PATH)

Cleaning log path: /content/drive/MyDrive/NimbusPay_Ticket_Triage/cleaning_log.csv


In [25]:
import csv

confirmed_ids = {
    "tr-00031", "tr-00050", "tr-00072",
    "tr-00107", "tr-00246", "tr-00341",
    "tr-00362", "tr-00541", "tr-00901",
    "tr-00919", "tr-01242", "tr-01271",
    "tr-01290"
}

with open(CLEANING_LOG_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["id", "action", "reason"]
    )

    for record_id in confirmed_ids:
        writer.writerow({
            "id": record_id,
            "action": "fix",
            "reason": "Explicit third-time contact statement: Teesri baar bol raha hoon"
        })

print("13 corrections added to cleaning_log.csv")

13 corrections added to cleaning_log.csv


In [26]:
print("Total training records:", len(final_records))

corrected = 0

for row in final_records:
    if row["id"] in confirmed_ids:
        label = json.loads(row["messages"][2]["content"])
        if label["needs_human"] is True:
            corrected += 1

print("Confirmed corrected records:", corrected)

Total training records: 1360
Confirmed corrected records: 13


In [27]:
import json
import re

with open(CLEAN_PATH, "r", encoding="utf-8") as f:
    saved_records = [json.loads(line) for line in f if line.strip()]

CONTACT_PATTERNS = [
    r"\bpichla\s+ticket\b",
    r"\bprevious\s+ticket\b",
    r"\bearlier\s+ticket\b",
    r"\b(?:baar|bar)\s+(?:baar|bar)\b",
    r"\b(?:teesri|tisri|third)\s+baar\b",
    r"\b(?:second|third|fourth)\s+time\b",
    r"\b(?:second|third|fourth)\s+complaint\b",
    r"\bcontacted support before\b",
    r"\b(?:i have|i've)\s+(?:already\s+)?contacted support\b",
    r"\b(?:already|previously)\s+(?:contacted|written|emailed|complained)\b",
    r"\b(?:written|emailed|contacted)\s+\d+\s+times\b",
    r"\bagain and again\b",
    r"\bpehle bhi complaint\b",
    r"\bcomplained before\b"
]

contact_regex = re.compile("|".join(CONTACT_PATTERNS), re.IGNORECASE)

def remove_quoted_lines(ticket):
    return "\n".join(
        line for line in ticket.splitlines()
        if not line.strip().startswith(">")
    )

mismatches = []

for row in saved_records:
    label = json.loads(row["messages"][2]["content"])
    ticket = remove_quoted_lines(row["messages"][1]["content"])

    expected = (
        label["priority"] == "P1"
        or bool(contact_regex.search(ticket))
    )

    if label["needs_human"] != expected:
        mismatches.append({
            "id": row["id"],
            "priority": label["priority"],
            "current": label["needs_human"],
            "expected": expected
        })

print("=" * 55)
print("FINAL ESCALATION VERIFICATION")
print("=" * 55)
print("Records checked:", len(saved_records))
print("Remaining mismatches:", len(mismatches))

for item in mismatches:
    print(item)

if not mismatches:
    print("\nVERIFICATION PASSED")
else:
    print("\nVERIFICATION FAILED")

FINAL ESCALATION VERIFICATION
Records checked: 1360
Remaining mismatches: 0

VERIFICATION PASSED


In [28]:
import torch
import gc
import os
import json

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("QLoRA libraries imported successfully.")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
QLoRA libraries imported successfully.


In [29]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Model:", MODEL_NAME)
print("Quantization: 4-bit NF4")
print("Compute dtype: FP16")
print("Double quantization: Enabled")
print("Configuration created successfully.")

Model: Qwen/Qwen2.5-1.5B-Instruct
Quantization: 4-bit NF4
Compute dtype: FP16
Double quantization: Enabled
Configuration created successfully.


In [30]:
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully.")

print("\nLoading 4-bit model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("\nMODEL LOADED SUCCESSFULLY")
print("Model:", MODEL_NAME)
print("Parameters:", f"{model.num_parameters():,}")
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


MODEL LOADED SUCCESSFULLY
Model: Qwen/Qwen2.5-1.5B-Instruct
Parameters: 1,543,714,304
GPU memory allocated: 1.07 GB
GPU memory reserved: 1.12 GB


In [31]:
model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

print("Model prepared for QLoRA.")
print("Gradient checkpointing enabled.")
print("Model cache disabled for training.")

Model prepared for QLoRA.
Gradient checkpointing enabled.
Model cache disabled for training.


In [32]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [33]:
import json
import numpy as np

def get_formatted_length(row):
    messages = row["messages"]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    return len(formatted)

train_lengths = [
    get_formatted_length(row)
    for row in final_records
]

print("TRAINING SEQUENCE LENGTHS")
print("=" * 40)
print("Records:", len(train_lengths))
print("Minimum:", min(train_lengths))
print("Maximum:", max(train_lengths))
print("Mean:", round(np.mean(train_lengths), 2))
print("Median:", np.median(train_lengths))
print("90th percentile:", np.percentile(train_lengths, 90))
print("95th percentile:", np.percentile(train_lengths, 95))
print("99th percentile:", np.percentile(train_lengths, 99))

TRAINING SEQUENCE LENGTHS
Records: 1360
Minimum: 2
Maximum: 2
Mean: 2.0
Median: 2.0
90th percentile: 2.0
95th percentile: 2.0
99th percentile: 2.0


In [34]:
import numpy as np

def get_formatted_length(row):
    encoded = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True
    )

    input_ids = encoded["input_ids"]

    # Handle both batched and unbatched outputs
    if hasattr(input_ids, "shape"):
        if len(input_ids.shape) == 2:
            return int(input_ids.shape[1])
        return int(input_ids.shape[0])

    if isinstance(input_ids[0], list):
        return len(input_ids[0])

    return len(input_ids)


train_lengths = [
    get_formatted_length(row)
    for row in final_records
]

print("CORRECTED TRAINING SEQUENCE LENGTHS")
print("=" * 45)
print("Records:", len(train_lengths))
print("Minimum:", min(train_lengths))
print("Maximum:", max(train_lengths))
print("Mean:", round(np.mean(train_lengths), 2))
print("Median:", np.median(train_lengths))
print("90th percentile:", round(np.percentile(train_lengths, 90), 2))
print("95th percentile:", round(np.percentile(train_lengths, 95), 2))
print("99th percentile:", round(np.percentile(train_lengths, 99), 2))

CORRECTED TRAINING SEQUENCE LENGTHS
Records: 1360
Minimum: 107
Maximum: 2215
Mean: 212.22
Median: 168.0
90th percentile: 214.0
95th percentile: 225.05
99th percentile: 1865.2


In [35]:
long_records = sorted(
    zip(train_lengths, final_records),
    key=lambda x: x[0],
    reverse=True
)

print("TOP 10 LONGEST TRAINING RECORDS")
print("=" * 50)

for length, row in long_records[:10]:
    print(
        f"ID: {row['id']} | Tokens: {length}"
    )
    print(
        "Ticket:",
        row["messages"][1]["content"][:350].replace("\n", " ")
    )
    print("-" * 50)

TOP 10 LONGEST TRAINING RECORDS
ID: tr-00817 | Tokens: 2215
Ticket: Received: 2026-07-14 (Tue) Source: email  Subject: Help needed  Hi, Ek hi order ke do baar paise kat gaye, ek wapas kar do. Android app use kar raha hoon. Card se hua tha. Rakam ₹99,936 hai. Yeh 10-07-26 ko hua. Account mein abhi Rs 3,762.25 pade hain. Teesri baar bol raha hoon. Dhanyavaad.  Regards, Arjun Das  > On 27 Jun 2026, NimbusPay Support w
--------------------------------------------------
ID: tr-00459 | Tokens: 2211
Ticket: Received: 2026-06-29 (Mon) Source: email  Subject: Issue with my account  Hello ji, Shopping ka payment decline ho gaya lekin bank ne paise kaat liye. Txn id NP 2764858720. Total Rs 1,68,444.99 ka tha. App version 6.4.2 hai. Transaction date 27 Jun hai. Please help karo.  Regards, Rahul Verma  > On 22 Jun 2026, NimbusPay Support wrote: > Thank you 
--------------------------------------------------
ID: tr-01095 | Tokens: 2174
Ticket: Received: 2026-07-11 (Sat) Source: email  Subject: Issue

In [36]:
MAX_LENGTH = 2304

truncated_records = sum(
    length > MAX_LENGTH
    for length in train_lengths
)

print("SEQUENCE CONFIGURATION")
print("=" * 40)
print("Maximum sequence length:", MAX_LENGTH)
print("Longest training record:", max(train_lengths))
print("Records requiring truncation:", truncated_records)
print("Records retained completely:", len(train_lengths) - truncated_records)

assert truncated_records == 0

print("\nSequence length configuration confirmed.")

SEQUENCE CONFIGURATION
Maximum sequence length: 2304
Longest training record: 2215
Records requiring truncation: 0
Records retained completely: 1360

Sequence length configuration confirmed.


In [37]:
def prepare_example(row):
    messages = row["messages"]

    # Tokenize the complete conversation
    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    # Tokenize system + user prompt only
    prompt_ids = tokenizer.apply_chat_template(
        messages[:2],
        tokenize=True,
        add_generation_prompt=True
    )

    # Ensure prompt is an exact prefix of full conversation
    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError(
            f"Prompt alignment failed for {row['id']}"
        )

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }


# Test three examples
for row in final_records[:3]:
    example = prepare_example(row)

    supervised_tokens = sum(
        token != -100 for token in example["labels"]
    )

    print("\nID:", row["id"])
    print("Total tokens:", len(example["input_ids"]))
    print("Masked prompt tokens:", example["labels"].count(-100))
    print("Assistant training tokens:", supervised_tokens)
    print("Alignment: PASSED")

    print(
        "Assistant target:",
        tokenizer.decode(
            [
                token for token in example["labels"]
                if token != -100
            ],
            skip_special_tokens=False
        )[:400]
    )
    print("-" * 50)

ValueError: Prompt alignment failed for tr-00001

In [41]:
row = final_records[0]

full_result = tokenizer.apply_chat_template(
    row["messages"],
    tokenize=True,
    add_generation_prompt=False
)

prompt_result = tokenizer.apply_chat_template(
    row["messages"][:2],
    tokenize=True,
    add_generation_prompt=True
)

print("FULL RESULT")
print("Type:", type(full_result))
print("Value preview:", repr(full_result)[:500])

print("\nPROMPT RESULT")
print("Type:", type(prompt_result))
print("Value preview:", repr(prompt_result)[:500])

if isinstance(full_result, dict):
    print("\nFull result keys:", full_result.keys())
    print("input_ids type:", type(full_result["input_ids"]))

if isinstance(prompt_result, dict):
    print("\nPrompt result keys:", prompt_result.keys())
    print("input_ids type:", type(prompt_result["input_ids"]))

FULL RESULT
Type: <class 'transformers.tokenization_utils_base.BatchEncoding'>
Value preview: {'input_ids': [151644, 8948, 198, 2610, 525, 71493, 20595, 594, 11727, 2406, 424, 1614, 13, 4457, 279, 11727, 323, 470, 279, 2406, 424, 4718, 13, 151645, 198, 151644, 872, 198, 23260, 25, 220, 17, 15, 17, 21, 12, 15, 22, 12, 17, 20, 320, 34882, 340, 3608, 25, 906, 60535, 271, 13048, 11, 358, 614, 264, 1909, 705, 429, 3207, 944, 728, 1526, 13, 8429, 30, 1084, 572, 3351, 13, 14562, 25, 7602, 22798, 13, 3017, 4540, 374, 264, 3731, 8155, 7036, 220, 16, 17, 13, 1084, 374, 220, 19, 13, 22, 74, 304, 27

PROMPT RESULT
Type: <class 'transformers.tokenization_utils_base.BatchEncoding'>
Value preview: {'input_ids': [151644, 8948, 198, 2610, 525, 71493, 20595, 594, 11727, 2406, 424, 1614, 13, 4457, 279, 11727, 323, 470, 279, 2406, 424, 4718, 13, 151645, 198, 151644, 872, 198, 23260, 25, 220, 17, 15, 17, 21, 12, 15, 22, 12, 17, 20, 320, 34882, 340, 3608, 25, 906, 60535, 271, 13048, 11, 358, 614, 264, 1909,

In [42]:
row = final_records[0]

full_result = tokenizer.apply_chat_template(
    row["messages"],
    tokenize=True,
    add_generation_prompt=False
)

prompt_result = tokenizer.apply_chat_template(
    row["messages"][:2],
    tokenize=True,
    add_generation_prompt=True
)

# Extract actual token IDs directly
full_ids = full_result["input_ids"]
prompt_ids = prompt_result["input_ids"]

print("FULL CONVERSATION:")
print(tokenizer.decode(full_ids[:100]))

print("\nPROMPT ONLY:")
print(tokenizer.decode(prompt_ids[-50:]))

print("\nTOKEN LENGTHS")
print("Full:", len(full_ids))
print("Prompt:", len(prompt_ids))

# Compare token sequences
limit = min(len(full_ids), len(prompt_ids))

difference = next(
    (i for i in range(limit) if full_ids[i] != prompt_ids[i]),
    None
)

print("\nFIRST DIFFERENCE:", difference)

if difference is not None:
    print("\nFull around difference:")
    print(tokenizer.decode(full_ids[max(0, difference-10):difference+20]))

    print("\nPrompt around difference:")
    print(tokenizer.decode(prompt_ids[max(0, difference-10):difference+20]))
else:
    print("\nOne sequence is a prefix of the other.")

FULL CONVERSATION:
<|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-07-25 (Sat)
Source: app-chat

Hi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 2

PROMPT ONLY:
. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply.<|im_end|>
<|im_start|>assistant


TOKEN LENGTHS
Full: 215
Prompt: 133

FIRST DIFFERENCE: None

One sequence is a prefix of the other.


In [43]:
def prepare_example(row):
    messages = row["messages"]

    full_result = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    prompt_result = tokenizer.apply_chat_template(
        messages[:2],
        tokenize=True,
        add_generation_prompt=True
    )

    full_ids = full_result["input_ids"]
    prompt_ids = prompt_result["input_ids"]

    # Verify exact prefix alignment
    assert full_ids[:len(prompt_ids)] == prompt_ids, (
        f"Alignment failed: {row['id']}"
    )

    # Mask system and user tokens
    labels = (
        [-100] * len(prompt_ids)
        + full_ids[len(prompt_ids):]
    )

    assert len(labels) == len(full_ids)
    assert any(token != -100 for token in labels)

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }


# Test three examples
for row in final_records[:3]:
    example = prepare_example(row)

    target_ids = [
        token for token in example["labels"]
        if token != -100
    ]

    print("\nID:", row["id"])
    print("Total tokens:", len(example["input_ids"]))
    print("Masked prompt tokens:", example["labels"].count(-100))
    print("Assistant training tokens:", len(target_ids))
    print("Assistant target:")
    print(tokenizer.decode(target_ids, skip_special_tokens=False))
    print("Alignment: PASSED")
    print("-" * 60)


ID: tr-00001
Total tokens: 215
Masked prompt tokens: 133
Assistant training tokens: 82
Assistant target:
{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>

Alignment: PASSED
------------------------------------------------------------

ID: tr-00002
Total tokens: 118
Masked prompt tokens: 66
Assistant training tokens: 52
Assistant target:
{"category": "account_access", "priority": "P2", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}<|im_end|>

Alignment: PASSED
------------------------------------------------------------

ID: tr-00003
Total tokens: 209
Masked prompt tokens: 129
Assistant training tokens: 80
Assistant target:
{"category": "refund", "priority": "P2", "amount_paise": 1915100, "txn_date": "2026-09-02", "txn_id": "NP2416305516", "channel": null, "language": "hi-

In [44]:
from tqdm.auto import tqdm

processed_data = []

for row in tqdm(final_records, desc="Tokenizing training data"):
    example = prepare_example(row)

    # Ensure no sequence exceeds our selected limit
    assert len(example["input_ids"]) <= MAX_LENGTH

    processed_data.append(example)

print("\nDATASET PROCESSING COMPLETE")
print("=" * 45)
print("Total examples:", len(processed_data))
print("Maximum sequence length:", MAX_LENGTH)

print(
    "Total supervised tokens:",
    sum(
        sum(token != -100 for token in item["labels"])
        for item in processed_data
    )
)

print(
    "Examples with no assistant targets:",
    sum(
        all(token == -100 for token in item["labels"])
        for item in processed_data
    )
)

print("\nAll examples processed successfully.")

Tokenizing training data:   0%|          | 0/1360 [00:00<?, ?it/s]


DATASET PROCESSING COMPLETE
Total examples: 1360
Maximum sequence length: 2304
Total supervised tokens: 87285
Examples with no assistant targets: 0

All examples processed successfully.


In [45]:
from dataclasses import dataclass
import torch

@dataclass
class CausalLMCollator:
    tokenizer: object

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)

        input_ids = []
        attention_masks = []
        labels = []

        for item in features:
            padding_length = max_len - len(item["input_ids"])

            input_ids.append(
                item["input_ids"] +
                [self.tokenizer.pad_token_id] * padding_length
            )

            attention_masks.append(
                item["attention_mask"] +
                [0] * padding_length
            )

            labels.append(
                item["labels"] +
                [-100] * padding_length
            )

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long)
        }


data_collator = CausalLMCollator(tokenizer)

# Test with two examples
test_batch = data_collator(processed_data[:2])

print("BATCH SHAPES")
print("=" * 40)

for key, value in test_batch.items():
    print(key, value.shape)

print("\nPadding token:", tokenizer.pad_token_id)
print("Collator test passed.")

BATCH SHAPES
input_ids torch.Size([2, 215])
attention_mask torch.Size([2, 215])
labels torch.Size([2, 215])

Padding token: 151643
Collator test passed.


In [47]:
import json
from tqdm.auto import tqdm

baseline_predictions = []

print("Running zero-shot baseline on 200 dev examples...")

model.eval()

with model.disable_adapter():
    for row in tqdm(dev_data, desc="Zero-shot baseline"):

        messages = row["messages"][:2]

        # Create chat-formatted input
        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        )

        # Extract input IDs explicitly
        inputs = encoded["input_ids"].to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        generated_tokens = outputs[0, inputs.shape[1]:]

        raw_output = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        baseline_predictions.append({
            "id": row["id"],
            "output": raw_output
        })

BASELINE_PATH = f"{PROJECT_DIR}/experiments/baseline_dev.jsonl"

with open(BASELINE_PATH, "w", encoding="utf-8") as f:
    for prediction in baseline_predictions:
        f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

print("\nBASELINE COMPLETE")
print("Predictions:", len(baseline_predictions))
print("Saved to:", BASELINE_PATH)

print("\nSAMPLE PREDICTION:")
print(json.dumps(baseline_predictions[0], indent=2))

Running zero-shot baseline on 200 dev examples...


Zero-shot baseline:   0%|          | 0/200 [00:00<?, ?it/s]


BASELINE COMPLETE
Predictions: 200
Saved to: /content/drive/MyDrive/NimbusPay_Ticket_Triage/experiments/baseline_dev.jsonl

SAMPLE PREDICTION:
{
  "id": "dv-00001",
  "output": "```json\n{\n  \"status\": \"IN_PROGRESS\",\n  \"priority\": \"MEDIUM\",\n  \"type\": \"USER_INQUIRY\",\n  \"ticket_number\": \"#54047\",\n  \"source\": \"app-chat\",\n  \"received_at\": \"2026-08-28T13:00:00Z\",\n  \"subject\": \"Activate and use a card\"\n}\n```"
}


In [48]:
import subprocess

result = subprocess.run(
    [
        "python",
        f"{PROJECT_DIR}/score.py",
        "--help"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

print("Exit code:", result.returncode)

STDOUT:
usage: score.py [-h] --gold GOLD --pred PRED [--json]

options:
  -h, --help   show this help message and exit
  --gold GOLD
  --pred PRED
  --json       print machine-readable JSON


STDERR:

Exit code: 0


In [49]:
import subprocess

GOLD_PATH = f"{DATA_DIR}/dev.jsonl"

result = subprocess.run(
    [
        "python",
        f"{PROJECT_DIR}/score.py",
        "--gold", GOLD_PATH,
        "--pred", BASELINE_PATH,
        "--json"
    ],
    capture_output=True,
    text=True
)

print("ZERO-SHOT EVALUATION")
print("=" * 50)
print("STDOUT:")
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

print("\nExit code:", result.returncode)

ZERO-SHOT EVALUATION
STDOUT:
{
 "all": {
  "n": 200,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "categor

In [ ]:
from transformers import TrainingArguments

EXPERIMENT_DIR = f"{PROJECT_DIR}/experiments/qlora_clean_r16"

training_args = TrainingArguments(
    output_dir=EXPERIMENT_DIR,

    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    fp16=True,
    bf16=False,

    optim="adamw_bnb_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,

    gradient_checkpointing=True,

    logging_steps=20,
    logging_first_step=True,

    save_strategy="epoch",
    save_total_limit=2,

    report_to="none",
    remove_unused_columns=False,

    dataloader_num_workers=0,
    seed=42
)

print("TRAINING CONFIGURATION")
print("=" * 45)
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)
print("Optimizer:", training_args.optim)
print("Output directory:", training_args.output_dir)
print("\nTraining arguments created successfully.")

In [51]:
import inspect
from transformers import TrainingArguments

signature = inspect.signature(TrainingArguments.__init__)

print("TRANSFORMERS TRAINING ARGUMENTS")
print("=" * 55)
print("Supported parameters:\n")

for name, param in signature.parameters.items():
    if name != "self":
        print(name, "=", param.default)

TRANSFORMERS TRAINING ARGUMENTS
Supported parameters:

output_dir = None
per_device_train_batch_size = 8
num_train_epochs = 3.0
max_steps = -1
learning_rate = 5e-05
lr_scheduler_type = linear
lr_scheduler_kwargs = None
warmup_steps = 0
optim = adamw_torch_fused
optim_args = None
weight_decay = 0.0
adam_beta1 = 0.9
adam_beta2 = 0.999
adam_epsilon = 1e-08
optim_target_modules = None
gradient_accumulation_steps = 1
average_tokens_across_devices = True
max_grad_norm = 1.0
label_smoothing_factor = 0.0
bf16 = False
fp16 = False
bf16_full_eval = False
fp16_full_eval = False
tf32 = None
gradient_checkpointing = False
gradient_checkpointing_kwargs = None
torch_compile = False
torch_compile_backend = None
torch_compile_mode = None
use_liger_kernel = False
liger_kernel_config = None
use_cache = False
neftune_noise_alpha = None
torch_empty_cache_steps = None
auto_find_batch_size = False
logging_strategy = steps
logging_steps = 500
logging_first_step = False
log_on_each_node = True
logging_nan_inf_

In [52]:
from transformers import TrainingArguments

EXPERIMENT_DIR = f"{PROJECT_DIR}/experiments/qlora_clean_r16"

training_args = TrainingArguments(
    output_dir=EXPERIMENT_DIR,

    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=17,

    fp16=True,
    bf16=False,

    optim="adamw_bnb_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,

    gradient_checkpointing=True,

    logging_steps=20,
    logging_first_step=True,

    save_strategy="epoch",
    save_total_limit=2,

    report_to="none",
    remove_unused_columns=False,

    dataloader_num_workers=0,
    seed=42
)

print("TRAINING CONFIGURATION")
print("=" * 45)
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Learning rate:", training_args.learning_rate)
print("Warmup steps:", training_args.warmup_steps)
print("Optimizer:", training_args.optim)
print("FP16:", training_args.fp16)
print("Output:", training_args.output_dir)

print("\nConfiguration created successfully.")

TRAINING CONFIGURATION
Epochs: 2
Batch size: 1
Gradient accumulation: 8
Learning rate: 0.0002
Warmup steps: 17
Optimizer: OptimizerNames.ADAMW_BNB
FP16: True
Output: /content/drive/MyDrive/NimbusPay_Ticket_Triage/experiments/qlora_clean_r16

Configuration created successfully.


In [53]:
import inspect
from transformers import Trainer

print("TRAINER API")
print("=" * 45)

signature = inspect.signature(Trainer.__init__)

for name in signature.parameters:
    print(name)

print("\nTrainer imported successfully.")

TRAINER API
self
model
args
data_collator
train_dataset
eval_dataset
processing_class
model_init
compute_loss_func
compute_metrics
callbacks
optimizers
optimizer_cls_and_kwargs
preprocess_logits_for_metrics

Trainer imported successfully.


In [54]:
from datasets import Dataset

train_dataset = Dataset.from_list(processed_data)

print("TRAINING DATASET CREATED")
print("=" * 45)
print("Dataset:", train_dataset)
print("Number of examples:", len(train_dataset))
print("Columns:", train_dataset.column_names)

# Inspect one example
sample = train_dataset[0]

print("\nFIRST EXAMPLE")
print("Input tokens:", len(sample["input_ids"]))
print("Label tokens:", len(sample["labels"]))
print(
    "Supervised tokens:",
    sum(x != -100 for x in sample["labels"])
)

assert len(train_dataset) == 1360
assert set(train_dataset.column_names) == {
    "input_ids", "attention_mask", "labels"
}

print("\nDataset validation passed.")

TRAINING DATASET CREATED
Dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1360
})
Number of examples: 1360
Columns: ['input_ids', 'attention_mask', 'labels']

FIRST EXAMPLE
Input tokens: 215
Label tokens: 215
Supervised tokens: 82

Dataset validation passed.


In [55]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("TRAINER INITIALIZED")
print("=" * 45)
print("Training examples:", len(trainer.train_dataset))
print("Model:", MODEL_NAME)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Trainer ready:", trainer is not None)

TRAINER INITIALIZED
Training examples: 1360
Model: Qwen/Qwen2.5-1.5B-Instruct
Trainable parameters: 18464768
Trainer ready: True


In [57]:
# Use non-reentrant gradient checkpointing
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Select the longest example
longest_index = max(
    range(len(processed_data)),
    key=lambda i: len(processed_data[i]["input_ids"])
)

test_example = processed_data[longest_index]
test_batch = data_collator([test_example])

# Move tensors to the model's device
test_batch = {
    key: value.to(model.device)
    for key, value in test_batch.items()
}

print("TEST EXAMPLE")
print("Sequence length:", len(test_example["input_ids"]))
print("GPU:", torch.cuda.get_device_name(0))

# Forward pass
model.zero_grad(set_to_none=True)
model.train()

outputs = model(**test_batch)
loss = outputs.loss

print("\nFORWARD PASS")
print("Loss:", round(loss.item(), 4))
print("Loss finite:", torch.isfinite(loss).item())

# Backward pass
loss.backward()

gradient_count = sum(
    1 for p in model.parameters()
    if p.requires_grad and p.grad is not None
)

print("\nBACKWARD PASS")
print("Parameters receiving gradients:", gradient_count)

# Clear test gradients
model.zero_grad(set_to_none=True)

print(
    "\nGPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

assert torch.isfinite(loss).item()
assert gradient_count > 0

print("\nFORWARD AND BACKWARD TEST PASSED")

TEST EXAMPLE
Sequence length: 2215
GPU: Tesla T4

FORWARD PASS
Loss: 1.846
Loss finite: True

BACKWARD PASS
Parameters receiving gradients: 392

GPU memory allocated: 4.5 GB

FORWARD AND BACKWARD TEST PASSED


In [58]:
import time
import torch

# Clear gradients from the previous test
model.zero_grad(set_to_none=True)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("Starting Experiment 1...")
print("Training examples:", len(train_dataset))
print("Epochs:", training_args.num_train_epochs)
print("Estimated optimizer steps: 340")
print("\nTraining started. Please wait...\n")

start_time = time.time()

train_result = trainer.train()

training_minutes = (time.time() - start_time) / 60

print("\n" + "=" * 50)
print("EXPERIMENT 1 COMPLETED")
print("=" * 50)

print("Training time:", round(training_minutes, 2), "minutes")
print("Final training loss:", train_result.training_loss)

peak_memory = torch.cuda.max_memory_allocated() / 1024**3
print("Peak GPU memory:", round(peak_memory, 2), "GB")

print("\nTraining metrics:")
print(train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Starting Experiment 1...
Training examples: 1360
Epochs: 2
Estimated optimizer steps: 340

Training started. Please wait...



Step,Training Loss
1,2.031640
20,0.876029
40,0.066932
60,0.055471
80,0.045440
100,0.035581
120,0.029684
140,0.035727
160,0.029766
180,0.025387



EXPERIMENT 1 COMPLETED
Training time: 32.18 minutes
Final training loss: 0.07941599621930544
Peak GPU memory: 9.17 GB

Training metrics:
{'train_runtime': 1929.7801, 'train_samples_per_second': 1.409, 'train_steps_per_second': 0.176, 'total_flos': 4602221745788928.0, 'train_loss': 0.07941599621930544, 'epoch': 2.0}


In [59]:
import os
import json

ADAPTER_PATH = f"{PROJECT_DIR}/adapter/experiment1_r16"

os.makedirs(ADAPTER_PATH, exist_ok=True)

# Save trained LoRA adapter
model.save_pretrained(ADAPTER_PATH)

# Save tokenizer
tokenizer.save_pretrained(ADAPTER_PATH)

# Save training metrics
with open(
    f"{EXPERIMENT_DIR}/training_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(train_result.metrics, f, indent=4)

print("EXPERIMENT 1 SAVED")
print("=" * 45)
print("Adapter:", ADAPTER_PATH)
print("Tokenizer saved.")
print("Training metrics saved.")

print("\nSaved files:")
print(os.listdir(ADAPTER_PATH))

EXPERIMENT 1 SAVED
Adapter: /content/drive/MyDrive/NimbusPay_Ticket_Triage/adapter/experiment1_r16
Tokenizer saved.
Training metrics saved.

Saved files:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [60]:
import json
import torch
from tqdm.auto import tqdm

model.eval()

finetuned_predictions = []

print("Generating fine-tuned predictions on 200 dev examples...")

for row in tqdm(dev_data, desc="Fine-tuned evaluation"):

    messages = row["messages"][:2]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    inputs = encoded["input_ids"].to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0, inputs.shape[1]:]

    raw_output = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    finetuned_predictions.append({
        "id": row["id"],
        "output": raw_output
    })

FINETUNED_PATH = f"{PROJECT_DIR}/experiments/finetuned_exp1_dev.jsonl"

with open(FINETUNED_PATH, "w", encoding="utf-8") as f:
    for prediction in finetuned_predictions:
        f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

print("\nPREDICTION GENERATION COMPLETE")
print("Predictions:", len(finetuned_predictions))
print("Saved:", FINETUNED_PATH)

print("\nSAMPLE PREDICTION:")
print(json.dumps(finetuned_predictions[0], indent=2))

Generating fine-tuned predictions on 200 dev examples...


Fine-tuned evaluation:   0%|          | 0/200 [00:00<?, ?it/s]


PREDICTION GENERATION COMPLETE
Predictions: 200
Saved: /content/drive/MyDrive/NimbusPay_Ticket_Triage/experiments/finetuned_exp1_dev.jsonl

SAMPLE PREDICTION:
{
  "id": "dv-00001",
  "output": "{\"category\": \"other\", \"priority\": \"P3\", \"amount_paise\": null, \"txn_date\": null, \"txn_id\": null, \"channel\": null, \"language\": \"en\", \"needs_human\": true}"
}


In [61]:
import subprocess

result = subprocess.run(
    [
        "python",
        f"{PROJECT_DIR}/score.py",
        "--gold", GOLD_PATH,
        "--pred", FINETUNED_PATH,
        "--json"
    ],
    capture_output=True,
    text=True
)

print("FINE-TUNED MODEL EVALUATION")
print("=" * 55)
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

print("Exit code:", result.returncode)

FINE-TUNED MODEL EVALUATION
{
 "all": {
  "n": 200,
  "json_valid": 1.0,
  "exact_match": 0.795,
  "mean_field_acc": 0.965,
  "fields": {
   "category": 0.985,
   "priority": 0.875,
   "amount_paise": 0.97,
   "txn_date": 0.98,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.91
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 1.0,
  "exact_match": 0.75,
  "mean_field_acc": 0.9609375,
  "fields": {
   "category": 0.9375,
   "priority": 0.9375,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.8125
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 1.0,
  "exact_match": 1.0,
  "mean_field_acc": 1.0,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 1.0,
  "exact_match": 0.7454545454545455,
  "mean_f

In [62]:
import json

prediction_map = {
    item["id"]: item["output"]
    for item in finetuned_predictions
}

errors = []

for row in dev_data:
    record_id = row["id"]

    gold = json.loads(row["messages"][2]["content"])
    raw_prediction = prediction_map[record_id]

    try:
        predicted = json.loads(raw_prediction)
    except json.JSONDecodeError:
        predicted = {}

    differences = {
        key: {
            "expected": gold.get(key),
            "predicted": predicted.get(key)
        }
        for key in gold
        if gold.get(key) != predicted.get(key)
    }

    if differences:
        errors.append({
            "id": record_id,
            "ticket": row["messages"][1]["content"],
            "differences": differences
        })

print("TOTAL INCORRECT PREDICTIONS:", len(errors))
print("=" * 60)

for item in errors[:10]:
    print("\nID:", item["id"])
    print("TICKET:", item["ticket"][:500])
    print("INCORRECT FIELDS:")

    for field, values in item["differences"].items():
        print(
            f"  {field}: expected={values['expected']}, "
            f"predicted={values['predicted']}"
        )

    print("-" * 60)

TOTAL INCORRECT PREDICTIONS: 41

ID: dv-00001
TICKET: Received: 2026-08-28 (Fri)
Source: app-chat

Good morning, What needs to be done to activate and use a card? My earlier ticket number is #54047. Please look into it.
INCORRECT FIELDS:
  needs_human: expected=False, predicted=True
------------------------------------------------------------

ID: dv-00005
TICKET: Received: 2026-08-09 (Sun)
Source: ivr-transcript

[IVR transcript] Hey, I tried to make a transfer but it failed. Dated Aug 1. Total 49680 rupees. Ref: NP9512428472 Kindly resolve this asap.
INCORRECT FIELDS:
  priority: expected=P2, predicted=P3
------------------------------------------------------------

ID: dv-00006
TICKET: Received: 2026-07-02 (Thu)
Source: twitter

@NimbusPayCare The App shows a fraudulent payment made on my account. It was day before yesterday. You can check np 75453 79360. Amount: 1.3 lakh
INCORRECT FIELDS:
  txn_date: expected=2026-06-30, predicted=2026-07-01
----------------------------------------

In [63]:
import re
import json

ticket_reference_pattern = re.compile(
    r"\b(?:pichla|previous|earlier)\s+ticket\b",
    re.IGNORECASE
)

explicit_contact_pattern = re.compile(
    r"\b(?:teesri|tisri|third)\s+baar\b|"
    r"\b(?:second|third|fourth)\s+time\b|"
    r"\b(?:baar|bar)\s+(?:baar|bar)\b|"
    r"\bagain and again\b|"
    r"\bcontacted support before\b|"
    r"\balready contacted support\b|"
    r"\bsecond|third|fourth complaint\b|"
    r"\bcomplained before\b",
    re.IGNORECASE
)

suspected_false_escalations = []

for row in final_records:
    label = json.loads(row["messages"][2]["content"])
    ticket = remove_quoted_lines(row["messages"][1]["content"])

    if (
        label["needs_human"] is True
        and label["priority"] != "P1"
        and ticket_reference_pattern.search(ticket)
        and not explicit_contact_pattern.search(ticket)
    ):
        suspected_false_escalations.append({
            "id": row["id"],
            "priority": label["priority"],
            "ticket": ticket[:500]
        })

print("SUSPECTED FALSE ESCALATIONS")
print("=" * 55)
print("Total:", len(suspected_false_escalations))

for item in suspected_false_escalations[:20]:
    print("\nID:", item["id"])
    print("Priority:", item["priority"])
    print("Ticket:", item["ticket"])
    print("-" * 55)

SUSPECTED FALSE ESCALATIONS
Total: 57

ID: tr-00005
Priority: P2
Ticket: Received: 2026-06-22 (Mon)
Source: twitter

@NimbusPayCare OTP nahi aa raha isliye sign in nahi kar pa raha. Pichla ticket number #88470 tha.
-------------------------------------------------------

ID: tr-00040
Priority: P3
Ticket: Received: 2026-06-08 (Mon)
Source: app-chat

My scratch card reward has not been added. My earlier ticket number is #78055. 331 rupees is the amount. Thanks.
-------------------------------------------------------

ID: tr-00044
Priority: P2
Ticket: Received: 2026-06-28 (Sun)
Source: app-chat

URGENT!! I'm locked out of my account as I can't remember my code, what do I do now? My earlier ticket number is #51075. Please help.
-------------------------------------------------------

ID: tr-00140
Priority: P2
Ticket: Received: 2026-06-17 (Wed)
Source: app-chat

Hello, I don't know my password anymore. I am using app version 5.9.5. My earlier ticket number is #80678. Thanks.
---------------

In [64]:
import json
import re
from collections import Counter

# Load the untouched original training file
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    original_records = [json.loads(line) for line in f if line.strip()]

original_map = {row["id"]: row for row in original_records}
cleaned_map = {row["id"]: row for row in final_records}

comparison = []

for item in suspected_false_escalations:
    rid = item["id"]

    original = original_map.get(rid)
    cleaned = cleaned_map.get(rid)

    if original is None or cleaned is None:
        continue

    try:
        original_label = json.loads(original["messages"][2]["content"])
        cleaned_label = json.loads(cleaned["messages"][2]["content"])

        comparison.append({
            "id": rid,
            "original": original_label.get("needs_human"),
            "cleaned": cleaned_label.get("needs_human"),
            "priority": cleaned_label.get("priority"),
            "changed": (
                original_label.get("needs_human")
                != cleaned_label.get("needs_human")
            )
        })
    except (json.JSONDecodeError, KeyError, IndexError, TypeError):
        continue

print("COMPARISON SUMMARY")
print("=" * 50)
print("Compared records:", len(comparison))
print("Original vs cleaned:", Counter(
    (x["original"], x["cleaned"]) for x in comparison
))

print("\nRECORDS WHERE CLEANING CHANGED needs_human")
print("=" * 50)

for x in comparison:
    if x["changed"]:
        print(
            f'{x["id"]}: original={x["original"]}, '
            f'cleaned={x["cleaned"]}, priority={x["priority"]}'
        )

print("\nRECORDS WHERE ORIGINAL AND CLEANED BOTH SAY TRUE")
print("=" * 50)

for x in comparison:
    if x["original"] is True and x["cleaned"] is True:
        print(x["id"])

COMPARISON SUMMARY
Compared records: 57
Original vs cleaned: Counter({(False, True): 57})

RECORDS WHERE CLEANING CHANGED needs_human
tr-00005: original=False, cleaned=True, priority=P2
tr-00040: original=False, cleaned=True, priority=P3
tr-00044: original=False, cleaned=True, priority=P2
tr-00140: original=False, cleaned=True, priority=P2
tr-00158: original=False, cleaned=True, priority=P2
tr-00173: original=False, cleaned=True, priority=P3
tr-00222: original=False, cleaned=True, priority=P3
tr-00285: original=False, cleaned=True, priority=P2
tr-00299: original=False, cleaned=True, priority=P2
tr-00305: original=False, cleaned=True, priority=P2
tr-00311: original=False, cleaned=True, priority=P3
tr-00340: original=False, cleaned=True, priority=P2
tr-00354: original=False, cleaned=True, priority=P3
tr-00361: original=False, cleaned=True, priority=P2
tr-00413: original=False, cleaned=True, priority=P3
tr-00428: original=False, cleaned=True, priority=P3
tr-00467: original=False, cleaned=

In [65]:
import json
from collections import Counter

false_to_true = []
true_to_false = []

for rid, cleaned in cleaned_map.items():
    original = original_map.get(rid)

    if original is None:
        continue

    try:
        old_label = json.loads(original["messages"][2]["content"])
        new_label = json.loads(cleaned["messages"][2]["content"])

        old_flag = old_label.get("needs_human")
        new_flag = new_label.get("needs_human")
        priority = new_label.get("priority")

        if old_flag is False and new_flag is True:
            false_to_true.append({
                "id": rid,
                "priority": priority,
                "ticket": cleaned["messages"][1]["content"][:350]
            })

        elif old_flag is True and new_flag is False:
            true_to_false.append(rid)

    except (json.JSONDecodeError, KeyError, IndexError, TypeError):
        continue

print("COMPLETE ESCALATION CHANGE AUDIT")
print("=" * 55)
print("False -> True:", len(false_to_true))
print("True -> False:", len(true_to_false))
print("Priority distribution of False -> True:")
print(Counter(x["priority"] for x in false_to_true))

print("\nFALSE -> TRUE RECORDS WITH P1")
for x in false_to_true:
    if x["priority"] == "P1":
        print(x["id"], x["priority"])

print("\nFIRST 15 FALSE -> TRUE EXAMPLES")
for x in false_to_true[:15]:
    print("\n", x["id"], "|", x["priority"])
    print(x["ticket"])

COMPLETE ESCALATION CHANGE AUDIT
False -> True: 57
True -> False: 4
Priority distribution of False -> True:
Counter({'P3': 38, 'P2': 19})

FALSE -> TRUE RECORDS WITH P1

FIRST 15 FALSE -> TRUE EXAMPLES

 tr-00005 | P2
Received: 2026-06-22 (Mon)
Source: twitter

@NimbusPayCare OTP nahi aa raha isliye sign in nahi kar pa raha. Pichla ticket number #88470 tha.

 tr-00040 | P3
Received: 2026-06-08 (Mon)
Source: app-chat

My scratch card reward has not been added. My earlier ticket number is #78055. 331 rupees is the amount. Thanks.

 tr-00044 | P2
Received: 2026-06-28 (Sun)
Source: app-chat

URGENT!! I'm locked out of my account as I can't remember my code, what do I do now? My earlier ticket number is #51075. Please help.

 tr-00140 | P2
Received: 2026-06-17 (Wed)
Source: app-chat

Hello, I don't know my password anymore. I am using app version 5.9.5. My earlier ticket number is #80678. Thanks.

 tr-00158 | P2
Received: 2026-06-01 (Mon)
Source: twitter

@NimbusPayCare I tried to pay with 

In [66]:
print("INSPECTING THE 4 TRUE -> FALSE CHANGES")
print("=" * 60)

for rid in true_to_false:
    original = original_map[rid]
    cleaned = cleaned_map[rid]

    old_label = json.loads(original["messages"][2]["content"])
    new_label = json.loads(cleaned["messages"][2]["content"])

    print("\nID:", rid)
    print("Original priority:", old_label.get("priority", old_label.get("prio")))
    print("Cleaned priority:", new_label.get("priority"))
    print("Original needs_human:", old_label.get("needs_human"))
    print("Cleaned needs_human:", new_label.get("needs_human"))
    print("Ticket:")
    print(cleaned["messages"][1]["content"])
    print("-" * 60)

INSPECTING THE 4 TRUE -> FALSE CHANGES

ID: tr-00073
Original priority: P1
Cleaned priority: P3
Original needs_human: True
Cleaned needs_human: False
Ticket:
Received: 2026-07-15 (Wed)
Source: email

Subject: Problem with NimbusPay

URGENT!!
I'm sure I have enough money so why didn't the store accept my card? The txn id is NP 4271342441. The transaction amount is Rs 1,68,787.25. Waiting for your reply.

Regards,
Sneha Iyer
------------------------------------------------------------

ID: tr-00187
Original priority: P1
Cleaned priority: P3
Original needs_human: True
Cleaned needs_human: False
Ticket:
Received: 2026-08-04 (Tue)
Source: email

Subject: Problem with NimbusPay

Hello ji,
Paise bheje the dost ko, usko mile nahi aur mere account se kat gaye. Txn id NP1220851898. 3rd August 2026 ki transaction hai. UPI se hua tha. 2.2 lakh ka transaction tha. Jaldi solve karo please.

Regards,
Rahul Verma
------------------------------------------------------------

ID: tr-00790
Original prior

In [67]:
import json
import pandas as pd

# 1. Restore the 57 incorrectly escalated records
restored_ids = set(x["id"] for x in false_to_true)

# 2. Repair the two incorrectly downgraded P1 records
p1_repair_ids = {"tr-00073", "tr-00187"}

correction_log = []

for row in final_records:
    rid = row["id"]

    if rid not in restored_ids and rid not in p1_repair_ids:
        continue

    label = json.loads(row["messages"][2]["content"])

    if rid in restored_ids:
        label["needs_human"] = False
        reason = "Restored original False label; earlier ticket reference alone does not establish repeated support contact."

    elif rid in p1_repair_ids:
        label["priority"] = "P1"
        label["needs_human"] = True
        reason = "Corrected priority to P1 because disputed transaction amount exceeds Rs 50,000."

    row["messages"][2]["content"] = json.dumps(
        label, ensure_ascii=False
    )

    correction_log.append({
        "id": rid,
        "action": "fix",
        "reason": reason
    })

# 3. Save corrected training data
with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    for row in final_records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# 4. Append corrections to cleaning log without duplicating these entries
log_df = pd.read_csv(CLEANING_LOG_PATH)

new_log_df = pd.DataFrame(correction_log)
existing_pairs = set(zip(log_df["id"], log_df["reason"]))

new_log_df = new_log_df[
    ~new_log_df.apply(
        lambda x: (x["id"], x["reason"]) in existing_pairs,
        axis=1
    )
]

log_df = pd.concat([log_df, new_log_df], ignore_index=True)
log_df.to_csv(CLEANING_LOG_PATH, index=False)

print("CORRECTION COMPLETE")
print("=" * 45)
print("Restored false escalation labels:", len(restored_ids))
print("Repaired P1 records:", len(p1_repair_ids))
print("Test-like records left unchanged:", 2)
print("New cleaning log entries:", len(new_log_df))
print("Corrected training records:", len(final_records))
print("Training data saved:", CLEAN_PATH)
print("Cleaning log saved:", CLEANING_LOG_PATH)

CORRECTION COMPLETE
Restored false escalation labels: 57
Repaired P1 records: 2
Test-like records left unchanged: 2
New cleaning log entries: 59
Corrected training records: 1360
Training data saved: /content/drive/MyDrive/NimbusPay_Ticket_Triage/data/train_cleaned.jsonl
Cleaning log saved: /content/drive/MyDrive/NimbusPay_Ticket_Triage/cleaning_log.csv


In [68]:
import json
import pandas as pd

# Reload the saved dataset, rather than relying only on notebook memory
with open(CLEAN_PATH, "r", encoding="utf-8") as f:
    verified_records = [json.loads(line) for line in f if line.strip()]

verified_map = {r["id"]: r for r in verified_records}

# Check all 57 restored records
restored_check = sum(
    json.loads(verified_map[rid]["messages"][2]["content"])["needs_human"] is False
    for rid in restored_ids
)

# Check both repaired P1 records
p1_check = {}
for rid in p1_repair_ids:
    label = json.loads(verified_map[rid]["messages"][2]["content"])
    p1_check[rid] = (
        label["priority"] == "P1"
        and label["needs_human"] is True
    )

# Check every P1 record has needs_human=True
p1_records = [
    r for r in verified_records
    if json.loads(r["messages"][2]["content"])["priority"] == "P1"
]

p1_consistency = all(
    json.loads(r["messages"][2]["content"])["needs_human"] is True
    for r in p1_records
)

# Verify schema key counts
expected_keys = {
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
}

schema_check = all(
    set(json.loads(r["messages"][2]["content"]).keys()) == expected_keys
    for r in verified_records
)

log_df = pd.read_csv(CLEANING_LOG_PATH)

print("FINAL DATA VERIFICATION")
print("=" * 50)
print("Records loaded:", len(verified_records))
print("Restored labels verified:", restored_check, "/", len(restored_ids))
print("P1 repairs verified:", p1_check)
print("All P1 records escalated:", p1_consistency)
print("Exact 8-key schema:", schema_check)
print("Cleaning log rows:", len(log_df))
print("Unique cleaning-log IDs:", log_df["id"].nunique())
print("Overall verification:",
      "PASS" if (
          len(verified_records) == 1360
          and restored_check == 57
          and all(p1_check.values())
          and p1_consistency
          and schema_check
      ) else "CHECK REQUIRED")

FINAL DATA VERIFICATION
Records loaded: 1360
Restored labels verified: 57 / 57
P1 repairs verified: {'tr-00187': True, 'tr-00073': True}
All P1 records escalated: True
Exact 8-key schema: True
Cleaning log rows: 446
Unique cleaning-log IDs: 325
Overall verification: PASS


In [69]:
from datasets import Dataset
import json

# Reload corrected records from Drive
with open(CLEAN_PATH, "r", encoding="utf-8") as f:
    final_records = [json.loads(line) for line in f if line.strip()]

# Regenerate tokenized examples using the existing assistant-only masking function
processed_data = [prepare_example(row) for row in final_records]

train_dataset = Dataset.from_list(processed_data)

# Verify the regenerated dataset
print("EXPERIMENT 2 DATASET")
print("=" * 45)
print("Training records:", len(train_dataset))
print("Dataset columns:", train_dataset.column_names)
print("Maximum sequence length:",
      max(len(x["input_ids"]) for x in processed_data))
print("Total supervised tokens:",
      sum(sum(t != -100 for t in x["labels"]) for x in processed_data))
print("Corrected records loaded:", len(final_records))

assert len(train_dataset) == 1360
assert all(
    len(x["input_ids"]) == len(x["labels"])
    for x in processed_data
)
assert all(
    len(x["input_ids"]) <= MAX_LENGTH
    for x in processed_data
)

print("\nDataset verification: PASS")

EXPERIMENT 2 DATASET
Training records: 1360
Dataset columns: ['input_ids', 'attention_mask', 'labels']
Maximum sequence length: 2215
Total supervised tokens: 87285
Corrected records loaded: 1360

Dataset verification: PASS


In [70]:
import os
import gc
import torch

from transformers import AutoModelForCausalLM
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Release Experiment 1 from GPU memory
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

# Experiment 2 directory
EXP2_DIR = f"{PROJECT_DIR}/experiments/qlora_clean_r8"
os.makedirs(EXP2_DIR, exist_ok=True)

# Load a fresh 4-bit base model
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare for QLoRA training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

model.config.use_cache = False

# Rank-8 LoRA
lora_config_exp2 = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config_exp2)

# Report configuration and memory
model.print_trainable_parameters()

print("\nEXPERIMENT 2 MODEL")
print("=" * 45)
print("Base model: Qwen2.5-1.5B-Instruct")
print("Quantization: 4-bit NF4")
print("LoRA rank: 8")
print("LoRA alpha: 16")
print("Learning rate target: 2e-4")
print("Epochs target: 2")
print("GPU allocated:", round(
    torch.cuda.memory_allocated() / 1024**3, 2
), "GB")
print("GPU reserved:", round(
    torch.cuda.memory_reserved() / 1024**3, 2
), "GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945

EXPERIMENT 2 MODEL
Base model: Qwen2.5-1.5B-Instruct
Quantization: 4-bit NF4
LoRA rank: 8
LoRA alpha: 16
Learning rate target: 2e-4
Epochs target: 2
GPU allocated: 4.82 GB
GPU reserved: 5.91 GB


In [71]:
from transformers import Trainer, TrainingArguments
import inspect

training_args_exp2 = TrainingArguments(
    output_dir=EXP2_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=17,
    fp16=True,
    bf16=False,
    optim="adamw_bnb_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    logging_steps=20,
    logging_first_step=True,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args_exp2,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Experiment 2 trainer initialized.")
print("Epochs:", training_args_exp2.num_train_epochs)
print("Learning rate:", training_args_exp2.learning_rate)
print("Effective batch size:",
      training_args_exp2.per_device_train_batch_size
      * training_args_exp2.gradient_accumulation_steps)
print("Expected optimizer steps:", len(trainer.get_train_dataloader()) // 8 * 2)

Experiment 2 trainer initialized.
Epochs: 2
Learning rate: 0.0002
Effective batch size: 8
Expected optimizer steps: 340


In [72]:
import time
import torch
import json
import os

# Reset peak memory statistics
torch.cuda.reset_peak_memory_stats()

# Start timer
exp2_start = time.time()

print("Starting Experiment 2 training...")
print("QLoRA | Rank 8 | LR 2e-4 | 2 Epochs")
print("=" * 50)

exp2_train_result = trainer.train()

# Capture runtime and peak GPU memory
exp2_runtime = time.time() - exp2_start
exp2_peak_memory = torch.cuda.max_memory_allocated() / 1024**3

exp2_metrics = {
    "experiment": "Experiment 2",
    "model": "Qwen2.5-1.5B-Instruct",
    "quantization": "4-bit NF4",
    "lora_rank": 8,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "epochs": 2,
    "training_runtime_seconds": exp2_runtime,
    "training_runtime_minutes": exp2_runtime / 60,
    "peak_gpu_memory_gb": exp2_peak_memory,
    "training_loss": exp2_train_result.training_loss,
    "train_samples": len(train_dataset)
}

with open(
    os.path.join(EXP2_DIR, "training_metrics.json"),
    "w"
) as f:
    json.dump(exp2_metrics, f, indent=4)

print("\nEXPERIMENT 2 TRAINING COMPLETE")
print("=" * 50)
print(json.dumps(exp2_metrics, indent=4))

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Starting Experiment 2 training...
QLoRA | Rank 8 | LR 2e-4 | 2 Epochs


Step,Training Loss
1,2.031611
20,1.106642
40,0.090495
60,0.058900
80,0.046719
100,0.038288
120,0.032774
140,0.042033
160,0.036255
180,0.030201



EXPERIMENT 2 TRAINING COMPLETE
{
    "experiment": "Experiment 2",
    "model": "Qwen2.5-1.5B-Instruct",
    "quantization": "4-bit NF4",
    "lora_rank": 8,
    "lora_alpha": 16,
    "learning_rate": 0.0002,
    "epochs": 2,
    "training_runtime_seconds": 1957.7313559055328,
    "training_runtime_minutes": 32.62885593175888,
    "peak_gpu_memory_gb": 9.438401699066162,
    "training_loss": 0.099088524764075,
    "train_samples": 1360
}


In [73]:
import os

EXP2_ADAPTER_PATH = f"{PROJECT_DIR}/adapter/experiment2_r8"
os.makedirs(EXP2_ADAPTER_PATH, exist_ok=True)

model.save_pretrained(EXP2_ADAPTER_PATH)
tokenizer.save_pretrained(EXP2_ADAPTER_PATH)

print("Experiment 2 adapter saved.")
print("Location:", EXP2_ADAPTER_PATH)
print("Files:", os.listdir(EXP2_ADAPTER_PATH))

Experiment 2 adapter saved.
Location: /content/drive/MyDrive/NimbusPay_Ticket_Triage/adapter/experiment2_r8
Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [75]:
import json
import os
import torch
import time

EXP2_PREDICTIONS_PATH = (
    f"{PROJECT_DIR}/experiments/experiment2_r8_dev.jsonl"
)

model.eval()
exp2_predictions = []

eval_start = time.time()

for i, row in enumerate(dev_data):
    encoded = tokenizer.apply_chat_template(
        row["messages"][:2],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    inputs = encoded["input_ids"].to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0, inputs.shape[1]:]
    raw_output = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    exp2_predictions.append({
        "id": row["id"],
        "output": raw_output
    })

    if (i + 1) % 50 == 0:
        print(f"Evaluated {i + 1}/200")

with open(EXP2_PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    for item in exp2_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("\nExperiment 2 evaluation complete.")
print("Predictions:", len(exp2_predictions))
print("Evaluation time:", round(time.time() - eval_start, 2), "seconds")
print("Saved:", EXP2_PREDICTIONS_PATH)
print("\nFirst prediction:")
print(json.dumps(exp2_predictions[0], indent=2, ensure_ascii=False))

Evaluated 50/200
Evaluated 100/200
Evaluated 150/200
Evaluated 200/200

Experiment 2 evaluation complete.
Predictions: 200
Evaluation time: 1392.81 seconds
Saved: /content/drive/MyDrive/NimbusPay_Ticket_Triage/experiments/experiment2_r8_dev.jsonl

First prediction:
{
  "id": "dv-00001",
  "output": "{\"category\": \"other\", \"priority\": \"P3\", \"amount_paise\": null, \"txn_date\": null, \"txn_id\": null, \"channel\": null, \"language\": \"en\", \"needs_human\": false}"
}


In [76]:
import subprocess

result = subprocess.run(
    [
        "python", "score.py",
        "--gold", DEV_PATH,
        "--pred", EXP2_PREDICTIONS_PATH,
        "--json"
    ],
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("\nOFFICIAL SCORING RESULTS:")
print(result.stdout)

if result.stderr:
    print("\nERROR / WARNINGS:")
    print(result.stderr)

RETURN CODE: 2

OFFICIAL SCORING RESULTS:


ERROR / WARNINGS:
python3: can't open file '/content/score.py': [Errno 2] No such file or directory



In [77]:
import os

print("Searching for score.py...")

for root, dirs, files in os.walk("/content/drive/MyDrive/NimbusPay_Ticket_Triage"):
    if "score.py" in files:
        print("FOUND:", os.path.join(root, "score.py"))

print("\nCurrent directory:", os.getcwd())
print("Project files:", os.listdir(PROJECT_DIR))

Searching for score.py...
FOUND: /content/drive/MyDrive/NimbusPay_Ticket_Triage/score.py

Current directory: /content
Project files: ['data', 'adapter', 'experiments', 'predictions', 'score.py', 'cleaning_log.csv']


In [78]:
result = subprocess.run(
    [
        "python",
        f"{PROJECT_DIR}/score.py",
        "--gold", DEV_PATH,
        "--pred", EXP2_PREDICTIONS_PATH,
        "--json"
    ],
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("\nEXPERIMENT 2 OFFICIAL SCORES:")
print(result.stdout)

if result.stderr:
    print("\nWARNINGS:")
    print(result.stderr)

RETURN CODE: 0

EXPERIMENT 2 OFFICIAL SCORES:
{
 "all": {
  "n": 200,
  "json_valid": 1.0,
  "exact_match": 0.565,
  "mean_field_acc": 0.92875,
  "fields": {
   "category": 0.98,
   "priority": 0.64,
   "amount_paise": 0.965,
   "txn_date": 0.985,
   "txn_id": 0.995,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.865
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 1.0,
  "exact_match": 0.9375,
  "mean_field_acc": 0.9921875,
  "fields": {
   "category": 1.0,
   "priority": 0.9375,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 1.0,
  "exact_match": 1.0,
  "mean_field_acc": 1.0,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 1.0,
  "exact_match": 0.38181818

In [79]:
import gc
import torch
import os

# Release Experiment 2 from GPU memory
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

# Reset experiment directory
EXP3_DIR = f"{PROJECT_DIR}/experiments/qlora_corrected_r16"
os.makedirs(EXP3_DIR, exist_ok=True)

# Load fresh 4-bit base model
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

model.config.use_cache = False

# Rank-16 LoRA
lora_config_exp3 = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config_exp3)
model.print_trainable_parameters()

print("Experiment 3 model loaded.")
print("GPU allocated:", round(
    torch.cuda.memory_allocated() / 1024**3, 2
), "GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Experiment 3 model loaded.
GPU allocated: 4.74 GB


In [80]:
from transformers import Trainer, TrainingArguments

training_args_exp3 = TrainingArguments(
    output_dir=EXP3_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=17,
    fp16=True,
    bf16=False,
    optim="adamw_bnb_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    logging_steps=20,
    logging_first_step=True,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args_exp3,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Experiment 3 trainer initialized.")
print("Training examples:", len(train_dataset))
print("Optimizer steps:", len(trainer.get_train_dataloader()) // 8 * 2)
print("Learning rate:", training_args_exp3.learning_rate)
print("Epochs:", training_args_exp3.num_train_epochs)

Experiment 3 trainer initialized.
Training examples: 1360
Optimizer steps: 340
Learning rate: 0.0002
Epochs: 2


In [81]:
import time
import torch
import json
import os

torch.cuda.reset_peak_memory_stats()
exp3_start = time.time()

print("Starting Experiment 3...")
print("QLoRA | Rank 16 | LR 2e-4 | 2 epochs")
print("=" * 50)

exp3_train_result = trainer.train()

exp3_runtime = time.time() - exp3_start
exp3_peak_memory = torch.cuda.max_memory_allocated() / 1024**3

exp3_metrics = {
    "experiment": "Experiment 3",
    "model": "Qwen2.5-1.5B-Instruct",
    "quantization": "4-bit NF4",
    "lora_rank": 16,
    "lora_alpha": 32,
    "learning_rate": 2e-4,
    "epochs": 2,
    "training_runtime_seconds": exp3_runtime,
    "training_runtime_minutes": exp3_runtime / 60,
    "peak_gpu_memory_gb": exp3_peak_memory,
    "training_loss": exp3_train_result.training_loss,
    "train_samples": len(train_dataset)
}

with open(os.path.join(EXP3_DIR, "training_metrics.json"), "w") as f:
    json.dump(exp3_metrics, f, indent=4)

print("\nEXPERIMENT 3 COMPLETE")
print(json.dumps(exp3_metrics, indent=4))

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Starting Experiment 3...
QLoRA | Rank 16 | LR 2e-4 | 2 epochs


Step,Training Loss
1,2.031611
20,0.883491
40,0.067758
60,0.053619
80,0.043257
100,0.033392
120,0.028943
140,0.036434
160,0.029638
180,0.024163



EXPERIMENT 3 COMPLETE
{
    "experiment": "Experiment 3",
    "model": "Qwen2.5-1.5B-Instruct",
    "quantization": "4-bit NF4",
    "lora_rank": 16,
    "lora_alpha": 32,
    "learning_rate": 0.0002,
    "epochs": 2,
    "training_runtime_seconds": 2000.743858575821,
    "training_runtime_minutes": 33.34573097626368,
    "peak_gpu_memory_gb": 9.414783000946045,
    "training_loss": 0.0791503624881015,
    "train_samples": 1360
}


In [82]:
EXP3_ADAPTER_PATH = f"{PROJECT_DIR}/adapter/experiment3_corrected_r16"
os.makedirs(EXP3_ADAPTER_PATH, exist_ok=True)

model.save_pretrained(EXP3_ADAPTER_PATH)
tokenizer.save_pretrained(EXP3_ADAPTER_PATH)

print("Experiment 3 adapter saved.")
print("Location:", EXP3_ADAPTER_PATH)
print("Files:", os.listdir(EXP3_ADAPTER_PATH))

Experiment 3 adapter saved.
Location: /content/drive/MyDrive/NimbusPay_Ticket_Triage/adapter/experiment3_corrected_r16
Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [83]:
import torch
import json
import time
import os

# Left padding is required for batched decoder-only generation
tokenizer.padding_side = "left"

def generate_batched(records, batch_size=4):
    predictions = []
    model.eval()

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        sequences = []

        for row in batch:
            encoded = tokenizer.apply_chat_template(
                row["messages"][:2],
                tokenize=True,
                add_generation_prompt=True
            )
            sequences.append(encoded["input_ids"])

        max_len = max(len(x) for x in sequences)

        input_ids = torch.full(
            (len(batch), max_len),
            tokenizer.pad_token_id,
            dtype=torch.long
        )
        attention_mask = torch.zeros_like(input_ids)

        for j, seq in enumerate(sequences):
            input_ids[j, -len(seq):] = torch.tensor(seq)
            attention_mask[j, -len(seq):] = 1

        input_ids = input_ids.to(model.device)
        attention_mask = attention_mask.to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        for j, row in enumerate(batch):
            generated = outputs[j, max_len:]
            raw_output = tokenizer.decode(
                generated,
                skip_special_tokens=True
            ).strip()

            predictions.append({
                "id": row["id"],
                "output": raw_output
            })

        if (start + len(batch)) % 20 == 0:
            print(
                f"Evaluated {min(start + len(batch), len(records))}"
                f"/{len(records)}"
            )

    return predictions


EXP3_PREDICTIONS_PATH = (
    f"{PROJECT_DIR}/experiments/experiment3_r16_dev.jsonl"
)

eval_start = time.time()
exp3_predictions = generate_batched(dev_data, batch_size=4)

with open(EXP3_PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    for item in exp3_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("\nExperiment 3 evaluation complete.")
print("Predictions:", len(exp3_predictions))
print("Time:", round((time.time() - eval_start) / 60, 2), "minutes")
print("Saved:", EXP3_PREDICTIONS_PATH)

Evaluated 20/200
Evaluated 40/200
Evaluated 60/200
Evaluated 80/200
Evaluated 100/200
Evaluated 120/200
Evaluated 140/200
Evaluated 160/200
Evaluated 180/200
Evaluated 200/200

Experiment 3 evaluation complete.
Predictions: 200
Time: 7.7 minutes
Saved: /content/drive/MyDrive/NimbusPay_Ticket_Triage/experiments/experiment3_r16_dev.jsonl


In [84]:
result = subprocess.run(
    [
        "python",
        f"{PROJECT_DIR}/score.py",
        "--gold", DEV_PATH,
        "--pred", EXP3_PREDICTIONS_PATH,
        "--json"
    ],
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("\nEXPERIMENT 3 OFFICIAL SCORES:")
print(result.stdout)

if result.stderr:
    print("\nWARNINGS:")
    print(result.stderr)

RETURN CODE: 0

EXPERIMENT 3 OFFICIAL SCORES:
{
 "all": {
  "n": 200,
  "json_valid": 1.0,
  "exact_match": 0.81,
  "mean_field_acc": 0.9662499999999999,
  "fields": {
   "category": 0.98,
   "priority": 0.86,
   "amount_paise": 0.965,
   "txn_date": 0.985,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.94
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 1.0,
  "exact_match": 0.9375,
  "mean_field_acc": 0.9921875,
  "fields": {
   "category": 0.9375,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 1.0,
  "exact_match": 1.0,
  "mean_field_acc": 1.0,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 1.0,
  "exact_match": 0.7

In [85]:
import json
import os
import time
import torch

tokenizer.padding_side = "left"
model.eval()

TEST_PREDICTIONS_PATH = (
    f"{PROJECT_DIR}/predictions/predictions_test.jsonl"
)

test_start = time.time()

test_predictions = generate_batched(test_data, batch_size=4)

# Save in the required submission format
with open(TEST_PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    for item in test_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

elapsed = time.time() - test_start

print("\nFINAL TEST PREDICTIONS GENERATED")
print("=" * 50)
print("Total predictions:", len(test_predictions))
print("Expected predictions:", 400)
print("Generation time:", round(elapsed / 60, 2), "minutes")
print("Saved to:", TEST_PREDICTIONS_PATH)

print("\nFirst prediction:")
print(json.dumps(test_predictions[0], indent=2, ensure_ascii=False))

Evaluated 20/400
Evaluated 40/400
Evaluated 60/400
Evaluated 80/400
Evaluated 100/400
Evaluated 120/400
Evaluated 140/400
Evaluated 160/400
Evaluated 180/400
Evaluated 200/400
Evaluated 220/400
Evaluated 240/400
Evaluated 260/400
Evaluated 280/400
Evaluated 300/400
Evaluated 320/400
Evaluated 340/400
Evaluated 360/400
Evaluated 380/400
Evaluated 400/400

FINAL TEST PREDICTIONS GENERATED
Total predictions: 400
Expected predictions: 400
Generation time: 15.12 minutes
Saved to: /content/drive/MyDrive/NimbusPay_Ticket_Triage/predictions/predictions_test.jsonl

First prediction:
{
  "id": "ts-00001",
  "output": "{\"category\": \"fraud\", \"priority\": \"P1\", \"amount_paise\": null, \"txn_date\": \"2026-07-00\", \"txn_id\": \"NP6090913363\", \"channel\": null, \"language\": \"hi-en\", \"needs_human\": true}"
}


In [86]:
import json
import re
from datetime import date
from collections import Counter

path = TEST_PREDICTIONS_PATH

with open(path, "r", encoding="utf-8") as f:
    saved_predictions = [json.loads(line) for line in f if line.strip()]

invalid_json = []
invalid_dates = []
invalid_schema = []
invalid_values = []

expected_keys = {
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
}

for item in saved_predictions:
    rid = item["id"]
    raw = item["output"]

    try:
        pred = json.loads(raw)
    except Exception:
        invalid_json.append(rid)
        continue

    if set(pred.keys()) != expected_keys:
        invalid_schema.append(rid)

    if pred.get("txn_date") is not None:
        try:
            date.fromisoformat(pred["txn_date"])
            if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", pred["txn_date"]):
                invalid_dates.append((rid, pred["txn_date"]))
        except (ValueError, TypeError):
            invalid_dates.append((rid, pred["txn_date"]))

    if pred.get("category") not in [
        "payment_failed", "refund", "fraud",
        "account_access", "kyc", "offers", "other"
    ]:
        invalid_values.append((rid, "category", pred.get("category")))

    if pred.get("priority") not in ["P1", "P2", "P3"]:
        invalid_values.append((rid, "priority", pred.get("priority")))

print("FINAL PREDICTION AUDIT")
print("=" * 50)
print("Total predictions:", len(saved_predictions))
print("Duplicate IDs:", len(saved_predictions) - len(set(x["id"] for x in saved_predictions)))
print("Invalid JSON:", len(invalid_json))
print("Invalid schema:", len(invalid_schema))
print("Invalid dates:", len(invalid_dates))
print("Invalid category/priority:", len(invalid_values))

print("\nINVALID DATE EXAMPLES:")
for x in invalid_dates[:15]:
    print(x)

print("\nFIRST PREDICTION:")
print(saved_predictions[0])

FINAL PREDICTION AUDIT
Total predictions: 400
Duplicate IDs: 0
Invalid JSON: 0
Invalid schema: 0
Invalid dates: 1
Invalid category/priority: 0

INVALID DATE EXAMPLES:
('ts-00001', '2026-07-00')

FIRST PREDICTION:
{'id': 'ts-00001', 'output': '{"category": "fraud", "priority": "P1", "amount_paise": null, "txn_date": "2026-07-00", "txn_id": "NP6090913363", "channel": null, "language": "hi-en", "needs_human": true}'}


In [87]:
import json

with open(EXP3_PREDICTIONS_PATH, "r", encoding="utf-8") as f:
    exp3_saved = [json.loads(line) for line in f if line.strip()]

gold_by_id = {
    row["id"]: json.loads(row["messages"][-1]["content"])
    for row in dev_data
}

errors = []

for item in exp3_saved:
    rid = item["id"]
    try:
        predicted = json.loads(item["output"])
    except Exception:
        errors.append((rid, "INVALID JSON", item["output"]))
        continue

    expected = gold_by_id[rid]
    wrong_fields = {
        key: {"expected": expected.get(key), "predicted": predicted.get(key)}
        for key in expected
        if expected.get(key) != predicted.get(key)
    }

    if wrong_fields:
        errors.append((rid, wrong_fields))

print("Experiment 3 errors:", len(errors), "out of", len(exp3_saved))
print("\nFIRST 10 ERRORS:")
for error in errors[:10]:
    print(error)

Experiment 3 errors: 38 out of 200

FIRST 10 ERRORS:
('dv-00005', {'priority': {'expected': 'P2', 'predicted': 'P3'}})
('dv-00011', {'priority': {'expected': 'P1', 'predicted': 'P2'}, 'needs_human': {'expected': True, 'predicted': False}})
('dv-00016', {'priority': {'expected': 'P2', 'predicted': 'P1'}, 'needs_human': {'expected': False, 'predicted': True}})
('dv-00021', {'priority': {'expected': 'P2', 'predicted': 'P3'}})
('dv-00025', {'amount_paise': {'expected': 2400, 'predicted': 24}})
('dv-00027', {'amount_paise': {'expected': 400000, 'predicted': 40000}})
('dv-00034', {'priority': {'expected': 'P2', 'predicted': 'P1'}, 'needs_human': {'expected': False, 'predicted': True}})
('dv-00037', {'priority': {'expected': 'P1', 'predicted': 'P3'}, 'needs_human': {'expected': True, 'predicted': False}})
('dv-00040', {'priority': {'expected': 'P2', 'predicted': 'P1'}, 'needs_human': {'expected': False, 'predicted': True}})
('dv-00053', {'priority': {'expected': 'P2', 'predicted': 'P3'}})


In [88]:
import json

dev_by_id = {row["id"]: row for row in dev_data}
pred_by_id = {item["id"]: item for item in exp3_saved}

for rid, _ in errors[:10]:
    row = dev_by_id[rid]
    predicted = json.loads(pred_by_id[rid]["output"])
    expected = json.loads(row["messages"][-1]["content"])

    user_message = "\n".join(
        msg["content"]
        for msg in row["messages"]
        if msg["role"] == "user"
    )

    print("\n" + "=" * 90)
    print("ID:", rid)
    print("TICKET:")
    print(user_message[:2500])
    print("\nEXPECTED:", expected)
    print("PREDICTED:", predicted)


ID: dv-00005
TICKET:
Received: 2026-08-09 (Sun)
Source: ivr-transcript

[IVR transcript] Hey, I tried to make a transfer but it failed. Dated Aug 1. Total 49680 rupees. Ref: NP9512428472 Kindly resolve this asap.

EXPECTED: {'category': 'payment_failed', 'priority': 'P2', 'amount_paise': 4968000, 'txn_date': '2026-08-01', 'txn_id': 'NP9512428472', 'channel': None, 'language': 'en', 'needs_human': False}
PREDICTED: {'category': 'payment_failed', 'amount_paise': 4968000, 'txn_date': '2026-08-01', 'txn_id': 'NP9512428472', 'channel': None, 'language': 'en', 'needs_human': False, 'priority': 'P3'}

ID: dv-00011
TICKET:
Received: 2026-06-17 (Wed)
Source: email

Subject: Problem with NimbusPay

URGENT!!
Why am I missing my refund. App version is 4.2.7. ₹84,137 was the value. This happened yesterday. ID: np3333133441

Regards,
Pooja Nair

EXPECTED: {'category': 'refund', 'priority': 'P1', 'amount_paise': 8413700, 'txn_date': '2026-06-16', 'txn_id': 'NP3333133441', 'channel': None, 'language'

In [89]:
import os
import shutil

source = EXP3_ADAPTER_PATH
destination = os.path.join(PROJECT_DIR, "adapter")

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors"
]

for filename in required_files:
    source_file = os.path.join(source, filename)
    assert os.path.exists(source_file), f"Missing: {source_file}"
    shutil.copy2(source_file, os.path.join(destination, filename))

# Copy tokenizer-related files if present
for filename in [
    "tokenizer_config.json",
    "tokenizer.json",
    "chat_template.jinja",
    "special_tokens_map.json"
]:
    source_file = os.path.join(source, filename)
    if os.path.exists(source_file):
        shutil.copy2(source_file, os.path.join(destination, filename))

print("FINAL ADAPTER FILES:")
for filename in sorted(os.listdir(destination)):
    path = os.path.join(destination, filename)
    if os.path.isfile(path):
        print(filename, f"({os.path.getsize(path) / (1024*1024):.2f} MB)")

print("\nAdapter config:", os.path.exists(os.path.join(destination, "adapter_config.json")))
print("Adapter weights:", os.path.exists(os.path.join(destination, "adapter_model.safetensors")))

FINAL ADAPTER FILES:
adapter_config.json (0.00 MB)
adapter_model.safetensors (70.49 MB)
chat_template.jinja (0.00 MB)
tokenizer.json (10.89 MB)
tokenizer_config.json (0.00 MB)

Adapter config: True
Adapter weights: True


In [90]:
report = r"""
# NimbusPay Support Ticket Triage: Fine-Tuning Report

## 1. Objective and approach

The objective was to fine-tune a small instruction model to classify NimbusPay support tickets and return exactly eight JSON fields. The base model was `Qwen/Qwen2.5-1.5B-Instruct` (approximately 1.54B parameters). Fine-tuning used PEFT QLoRA with 4-bit NF4 quantization on a Google Colab NVIDIA Tesla T4 GPU.

The final selected adapter is Experiment 3, using LoRA rank 16 and alpha 32. It has 18,464,768 trainable parameters (1.182% of the base model). The final adapter is provided in `adapter/`.

## 2. Data audit and preparation

The original training set contained 1,445 records. The audit identified malformed/truncated labels, legacy priority keys, priority inconsistencies, date-format issues, transaction-ID formatting issues, and duplicate messages.

| Audit finding | Records |
|---|---:|
| Malformed/truncated labels | 25 |
| Incorrect/legacy priority keys | 70 |
| Incorrect transaction dates | 30 |
| Incorrect transaction IDs | 20 |
| Duplicate records identified | 60 |
| Final cleaned training records | 1,360 |

Malformed labels were excluded rather than reconstructed. Duplicate records were removed. Legacy priority fields were migrated and priorities recalculated against the supplied schema. Dates and transaction IDs were normalized. Additional manual corrections restored 57 escalation labels that had been incorrectly changed by an earlier cleaning rule and repaired two P1 cases. The changes are recorded in `cleaning_log.csv`.

The final cleaned dataset contains 1,360 records. The 200-record development set was kept separate from training. The 400 test records were used only for final inference.

Sequence-length analysis gave a median of 168 tokens, 95th percentile of approximately 225 tokens, and maximum of 2,215 tokens. A maximum sequence length of 2,304 was selected, so no training example was truncated. The final tokenized dataset contained 87,285 supervised assistant tokens and zero empty assistant targets. Prompt tokens were masked with label `-100`; loss was computed only on assistant targets.

## 3. Baselines and experiments

The base-model zero-shot baseline produced 0% valid JSON, 0% exact match, and 0% mean field accuracy on the 200 development records. A separate base-model baseline with schema rules embedded in the prompt was not completed.

| Run | Training data | Configuration | Dev exact match | Mean field accuracy | Peak GPU memory | Training time |
|---|---|---|---:|---:|---:|---:|
| Experiment 1 | Earlier cleaned version | QLoRA, rank 16, 2 epochs | 79.5% | 96.50% | 9.17 GB | 32.18 min |
| Experiment 2 | Corrected cleaned data | QLoRA, rank 8, 2 epochs | 56.5% | 92.875% | 9.44 GB | 32.63 min |
| Experiment 3 | Corrected cleaned data | QLoRA, rank 16, 2 epochs | **81.0%** | **96.625%** | 9.415 GB | 33.35 min |

All fine-tuning runs used a learning rate of 2e-4, effective batch size 8, 340 optimizer steps, warmup of 17 steps, cosine scheduling, FP16 compute, gradient checkpointing, and 8-bit AdamW. Experiment 2 and Experiment 3 form the controlled rank comparison: rank 8 versus rank 16, with the corrected dataset and other settings held constant.

Experiment 1 used an earlier cleaned dataset containing incorrect escalation labels. Its score is retained for transparency but should not be treated as a fair comparison against Experiments 2 and 3.

The assignment requested at least four controlled experiments, including raw versus cleaned data and 16-bit LoRA versus 4-bit QLoRA. Those comparisons, a fourth experiment, and the prompt-rules baseline were not completed within the available compute time. No results are fabricated for these missing experiments.

## 4. Best model results and error analysis

Experiment 3 achieved 100% JSON validity, 81.0% exact match (162/200), and 96.625% mean field accuracy.

| Field | Accuracy |
|---|---:|
| Category | 98.0% |
| Priority | 86.0% |
| Amount (paise) | 96.5% |
| Transaction date | 98.5% |
| Transaction ID | 100.0% |
| Channel | 100.0% |
| Language | 100.0% |
| Needs human | 94.0% |

The strongest categories were KYC (100% exact match) and other (93.75%). Refund achieved 78.18%, payment_failed 71.70%, fraud 80.0%, offers 77.78%, and account_access 84.21%.

There were 38 incorrect predictions in the development set. The first ten observed failures were:

| Ticket | Incorrect field(s) | Observed error |
|---|---|---|
| dv-00005 | priority | P2 predicted as P3 |
| dv-00011 | priority, needs_human | P1 predicted as P2; true predicted false |
| dv-00016 | priority, needs_human | P2 predicted as P1; false predicted true |
| dv-00021 | priority | P2 predicted as P3 |
| dv-00025 | amount_paise | 2,400 predicted as 24 |
| dv-00027 | amount_paise | 400,000 predicted as 40,000 |
| dv-00034 | priority, needs_human | P2 predicted as P1; false predicted true |
| dv-00037 | priority, needs_human | P1 predicted as P3; true predicted false |
| dv-00040 | priority, needs_human | P2 predicted as P1; false predicted true |
| dv-00053 | priority | P2 predicted as P3 |

Inspection of the available ticket texts showed inconsistent handling of shorthand amount expressions such as `24` and `4k`, and inconsistent application of priority thresholds. Priority mistakes also propagated into `needs_human`, because P1 cases require human escalation. These examples indicate that numeric normalization and rule-boundary reasoning are the main areas for improvement. Transaction ID, channel, and language extraction were perfect on the development set.

## 5. Final test inference

The selected Experiment 3 adapter generated 400 predictions for the unlabelled test set. The saved file is `predictions/predictions_test.jsonl`.

| Validation check | Result |
|---|---:|
| Total predictions | 400 |
| Duplicate IDs | 0 |
| Invalid JSON | 0 |
| Invalid schema | 0 |
| Invalid category/priority | 0 |
| Invalid dates | 1 |

Ticket `ts-00001` contains the model-generated date `2026-07-00`, which is not a valid calendar date. The prediction was not manually altered because the inference instructions permit only whitespace stripping after generation. The original raw model output is retained in the submission.

## 6. Limitations and next steps

The experiment budget did not allow all requested controlled comparisons. The results are based on one development split, and the test set has no labels available for local scoring. Future work should prioritize a raw-versus-cleaned data comparison, 16-bit LoRA versus QLoRA, a learning-rate or epoch sweep, and a schema-rules baseline. Further error-focused data review should target priority thresholds, escalation conditions, and rupee-to-paise conversion.

## 7. AI assistance disclosure

ChatGPT was used for coding guidance, debugging assistance, analysis support, and documentation preparation. Training labels and test predictions were not generated by an external AI service. Test predictions were generated by the fine-tuned model using the specified inference process.
"""

report_path = os.path.join(PROJECT_DIR, "report.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report.strip() + "\n")

print("Report saved:", report_path)
print("Word count:", len(report.split()))
print("File size:", os.path.getsize(report_path), "bytes")

Report saved: /content/drive/MyDrive/NimbusPay_Ticket_Triage/report.md
Word count: 1098
File size: 7033 bytes


In [92]:
readme_lines = [
    "# NimbusPay Support Ticket Triage",
    "",
    "Fine-tuning a small instruction model for structured NimbusPay support-ticket classification.",
    "",
    "## Final model",
    "- Base model: Qwen/Qwen2.5-1.5B-Instruct",
    "- Fine-tuning: PEFT QLoRA, 4-bit NF4",
    "- LoRA rank: 16",
    "- LoRA alpha: 32",
    "- Trainable parameters: 18,464,768 (1.182%)",
    "- GPU: NVIDIA Tesla T4",
    "- Maximum sequence length: 2304",
    "- Training examples: 1,360",
    "",
    "## Results",
    "| Metric | Experiment 3 |",
    "|---|---:|",
    "| Development exact match | 81.0% |",
    "| Mean field accuracy | 96.625% |",
    "| JSON validity | 100% |",
    "| Peak GPU memory | 9.415 GB |",
    "| Training time | 33.35 minutes |",
    "",
    "See report.md for the data audit, experiments, error analysis, limitations, and AI assistance disclosure.",
    "",
    "## Repository structure",
    "```text",
    "NimbusPay_Ticket_Triage/",
    "├── adapter/",
    "│   ├── adapter_config.json",
    "│   ├── adapter_model.safetensors",
    "│   ├── tokenizer.json",
    "│   ├── tokenizer_config.json",
    "│   └── chat_template.jinja",
    "├── experiments/",
    "├── predictions/",
    "│   └── predictions_test.jsonl",
    "├── cleaning_log.csv",
    "├── notebook.ipynb",
    "├── README.md",
    "└── report.md",
    "```",
    "",
    "## Inference",
    "Load Qwen/Qwen2.5-1.5B-Instruct, load the PEFT adapter from adapter/, apply the model chat template, and generate greedily with do_sample=False and max_new_tokens=200. Strip surrounding whitespace only.",
    "",
    "The base model and original datasets are not included in this repository.",
    "",
    "## Limitations",
    "- The schema-rules prompt baseline was not completed.",
    "- Raw-versus-cleaned and 16-bit LoRA-versus-QLoRA comparisons were not completed.",
    "- Three fine-tuning runs were performed. Experiment 2 versus Experiment 3 is the controlled rank comparison.",
    "- Experiment 1 used an earlier cleaned dataset and is not directly comparable.",
    "- One of 400 test predictions contains the invalid model-generated date 2026-07-00. It was retained unchanged.",
    "",
    "## AI assistance disclosure",
    "ChatGPT was used for coding guidance, debugging, analysis support, and documentation preparation. Training and final inference were performed in the Colab environment."
]

readme_path = os.path.join(PROJECT_DIR, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write("\n".join(readme_lines) + "\n")

print("README saved:", readme_path)
print("File size:", os.path.getsize(readme_path), "bytes")

README saved: /content/drive/MyDrive/NimbusPay_Ticket_Triage/README.md
File size: 2079 bytes


In [93]:
import os
import json

required_paths = {
    "README": f"{PROJECT_DIR}/README.md",
    "Report": f"{PROJECT_DIR}/report.md",
    "Cleaning log": CLEANING_LOG_PATH,
    "Test predictions": TEST_PREDICTIONS_PATH,
    "Adapter config": f"{PROJECT_DIR}/adapter/adapter_config.json",
    "Adapter weights": f"{PROJECT_DIR}/adapter/adapter_model.safetensors",
}

print("FINAL SUBMISSION FILE AUDIT")
print("=" * 55)

all_present = True

for name, path in required_paths.items():
    exists = os.path.isfile(path)
    size = os.path.getsize(path) if exists else 0
    print(f"{'PASS' if exists else 'MISSING'} | {name}: {size:,} bytes")
    all_present = all_present and exists

print("\nPROJECT ROOT:")
for item in sorted(os.listdir(PROJECT_DIR)):
    print(" -", item)

# Verify test prediction count
with open(TEST_PREDICTIONS_PATH, "r", encoding="utf-8") as f:
    test_count = sum(1 for line in f if line.strip())

print("\nTest prediction count:", test_count)
print("Expected count: 400")

# Check for notebook in project root
notebook_path = f"{PROJECT_DIR}/notebook.ipynb"
print("\nNotebook saved in project:", os.path.isfile(notebook_path))

print("\nOVERALL FILE CHECK:", "PASS" if all_present and test_count == 400 else "CHECK REQUIRED")

FINAL SUBMISSION FILE AUDIT
PASS | README: 2,079 bytes
PASS | Report: 7,033 bytes
PASS | Cleaning log: 37,546 bytes
PASS | Test predictions: 89,706 bytes
PASS | Adapter config: 1,182 bytes
PASS | Adapter weights: 73,911,112 bytes

PROJECT ROOT:
 - README.md
 - adapter
 - cleaning_log.csv
 - data
 - experiments
 - predictions
 - report.md
 - score.py

Test prediction count: 400
Expected count: 400

Notebook saved in project: False

OVERALL FILE CHECK: PASS
